# L1+L6: Perfect Metal + Surface Kinetics — Analysis Notebook

**Physics:**
- Surface dissociation/recombination at the gas–metal interface (L6)
- Bulk lattice diffusion of atomic H through a perfect metal (L1)
- No oxide layer

**Two competing resistances:**
- $R_{surface} \sim 1 / (k_{diss} \cdot P_{up} \cdot (1-\theta)^2)$
- $R_{metal} = L_m / (D_m \cdot K_{s,m})$

**Key physics question:** At what pressure and temperature does the rate-limiting step shift from surface dissociation to bulk diffusion?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os
import sys

# Add parent directory to path
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
# This notebook must run with its OWN folder as the working directory: that is how
# calculations.config.model_config finds this folder's study_config.py, and how
# RESULTS_DIR below resolves. Fails loudly rather than silently using another study.
from calculations.config import assert_study_is_local
print(assert_study_is_local())

# Figures from this notebook go in one folder beside it.
from calculations.figures import results_dir, save_panels
from calculations import surface_figures as SF   # the figures, in the house style
FIGS = results_dir("Surface_proposal")

In [ ]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import solve_steady_state_flux_L1L6
from calculations.config.model_config import (
    R, F,
    METALS, CONDITIONS,
    build_simulation_config,
)

# ── Simulation config ────────────────────────────────────────────────────────
SIM       = build_simulation_config(
)

METAL_KEY = SIM['metal_name']
METAL     = METALS[METAL_KEY]
SK        = METAL['surface_kinetics']     # surface kinetics sub-dict

# ── Pressure sweep (matches widget range) ────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)   #np.logspace(-3, 12, 100)
P_DOWN      = SIM['P_downstream']         # Pa
T_RANGE     = CONDITIONS['T_range']       # K
N_T         = CONDITIONS['n_points']

# ── Arrhenius helper functions ───────────────────────────────────────────────
def get_k_diss_at_T(T_K):
    """k_diss at temperature T_K via Arrhenius from config."""
    return SK['k_diss_metal_ref'] * np.exp(
        (-SK['E_diss_metal'] / R) * (1/T_K - 1/SK['T_ref'])
    )

def get_K_eq_at_T(T_K):
    """K_eq at temperature T_K via van't Hoff from config."""
    return SK['K_eq_metal_ref'] * np.exp(
        (-SK['H_eq_metal'] / R) * (1/T_K - 1/SK['T_ref'])
    )

def get_D_m_at_T(T_K):
    """D_m at temperature T_K via Arrhenius from config."""
    return METAL['D_ref'] * np.exp(
        (-METAL['E_D'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_K_s_m_at_T(T_K):
    """K_s_m at temperature T_K via Arrhenius from config."""
    return METAL['K_s_ref'] * np.exp(
        (-METAL['H_s'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

# ── Known reference activation energies (for validation) ────────────────────
E_REF_SURFACE = SK['E_diss_metal']              # J/mol — surface dissociation
E_REF_METAL   = METAL['E_D'] + METAL['H_s']    # J/mol — bulk permeation

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Metal:           {METAL_KEY}")
print(f"  E_ref surface:   {E_REF_SURFACE/1000:.3f} kJ/mol  (E_diss)")
print(f"  E_ref metal:     {E_REF_METAL/1000:.3f} kJ/mol  (E_D + H_s)")
print(f"  P range:         {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:         {T_RANGE[0]} – {T_RANGE[1]} K")

In [42]:
P_RANGE_ARR

array([1.00000000e-07, 2.28546386e-07, 5.22334507e-07, 1.19377664e-06,
       2.72833338e-06, 6.23550734e-06, 1.42510267e-05, 3.25702066e-05,
       7.44380301e-05, 1.70125428e-04, 3.88815518e-04, 8.88623816e-04,
       2.03091762e-03, 4.64158883e-03, 1.06081836e-02, 2.42446202e-02,
       5.54102033e-02, 1.26638017e-01, 2.89426612e-01, 6.61474064e-01,
       1.51177507e+00, 3.45510729e+00, 7.89652287e+00, 1.80472177e+01,
       4.12462638e+01, 9.42668455e+01, 2.15443469e+02, 4.92388263e+02,
       1.12533558e+03, 2.57191381e+03, 5.87801607e+03, 1.34339933e+04,
       3.07029063e+04, 7.01703829e+04, 1.60371874e+05, 3.66524124e+05,
       8.37677640e+05, 1.91448198e+06, 4.37547938e+06, 1.00000000e+07])

In [43]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change placeholder values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ─────────────────────────────────────────────────────
T_op  = SIM['T_operating']    # K
L_m   = SIM['L_metal']        # m
P_down = SIM['P_downstream']  # Pa

# ── Surface kinetics at T_op ─────────────────────────────────────────────────
k_diss = get_k_diss_at_T(T_op)
K_eq   = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq      # derived — not stored separately

# ── Metal transport at T_op ──────────────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Derived ──────────────────────────────────────────────────────────────────
beta = D_m * K_s_m / L_m      # metal permeance [mol/m²/s/Pa^0.5]

# ── Print verification ───────────────────────────────────────────────────────
print("=" * 60)
print(f"L1+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_m      = {L_m:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Surface kinetics  [config: surface_kinetics sub-dict]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s  (= k_diss / K_eq)")

print(f"\n  Metal transport  [config: Arrhenius]")
print(f"    D_m      = {D_m:.3e}  m²/s")
print(f"    K_s_m    = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Derived")
print(f"    β        = {beta:.3e}  mol/m²/s/Pa^0.5  (= D_m × K_s_m / L_m)")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss   = {E_REF_SURFACE/1000:.3f} kJ/mol   (surface dissociation)")
print(f"    E_D+Hs   = {E_REF_METAL/1000:.3f} kJ/mol   (bulk permeation)")

# ── Placeholder warning ──────────────────────────────────────────────────────
if SK['k_diss_metal_ref'] <= 1e-12:
    print(f"\n  ⚠  WARNING: k_diss_metal_ref = {SK['k_diss_metal_ref']:.0e} looks like")
    print(f"     a placeholder. Update config surface_kinetics sub-dict")
    print(f"     with literature values before interpreting results.")

print("\n" + "=" * 60)

L1+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_m      = 1.000e-03  m
    P_down   = 0.0e+00  Pa

  Surface kinetics  [config: surface_kinetics sub-dict]
    k_diss   = 4.611e-07  mol/m²/s/Pa
    K_eq     = 8.212e-04  Pa⁻¹
    k_recomb = 5.615e-04  mol/m²/s  (= k_diss / K_eq)

  Metal transport  [config: Arrhenius]
    D_m      = 2.487e-10  m²/s
    K_s_m    = 3.834e-02  mol/m³/Pa^0.5

  Derived
    β        = 9.534e-09  mol/m²/s/Pa^0.5  (= D_m × K_s_m / L_m)

  Reference activation energies  [config]
    E_diss   = 81.560 kJ/mol   (surface dissociation)
    E_D+Hs   = 79.400 kJ/mol   (bulk permeation)



In [44]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop ─────────────────────────────────────────────────────────
rows             = []
J_arr            = []
theta_arr        = []
P_int_arr        = []
frac_surface_arr = []
frac_metal_arr   = []
R_surface_arr    = []
R_metal_arr      = []
R_total_arr      = []
rate_lim_arr     = []

for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_L1L6(
            P_up, P_down, L_m,
            k_diss, K_eq, D_m, K_s_m
        )
        J_arr.append(r['J_ss'])
        theta_arr.append(r['theta'])
        P_int_arr.append(r['P_int'])
        frac_surface_arr.append(r['resistances']['fraction_surface'])
        frac_metal_arr.append(r['resistances']['fraction_metal'])
        R_surface_arr.append(r['resistances']['R_surface'])
        R_metal_arr.append(r['resistances']['R_metal'])
        R_total_arr.append(r['resistances']['R_total'])
        rate_lim_arr.append(r['rate_limiting'])

        rows.append({
            'P_up (Pa)':            P_up,
            'P_int (Pa)':           r['P_int'],
            'J_ss (mol/m²/s)':      r['J_ss'],
            'θ_surface':            r['theta'],
            'β_metal':              r['beta'],
            'R_surface':            r['resistances']['R_surface'],
            'R_metal':              r['resistances']['R_metal'],
            'R_total':              r['resistances']['R_total'],
            'fraction_surface (%)': r['resistances']['fraction_surface'] * 100,
            'fraction_metal (%)':   r['resistances']['fraction_metal']   * 100,
            'Rate-Limiting':        r['rate_limiting'].upper(),
        })

    except Exception as e:
        J_arr.append(np.nan);            theta_arr.append(np.nan)
        P_int_arr.append(np.nan);        frac_surface_arr.append(np.nan)
        frac_metal_arr.append(np.nan);   R_surface_arr.append(np.nan)
        R_metal_arr.append(np.nan);      R_total_arr.append(np.nan)
        rate_lim_arr.append('error')
        rows.append({'P_up (Pa)': P_up, 'Rate-Limiting': 'ERROR', 'Error': str(e)})

# ── Convert to arrays ────────────────────────────────────────────────────────
J_arr            = np.array(J_arr)
theta_arr        = np.array(theta_arr)
P_int_arr        = np.array(P_int_arr)
frac_surface_arr = np.array(frac_surface_arr)
frac_metal_arr   = np.array(frac_metal_arr)
R_surface_arr    = np.array(R_surface_arr)
R_metal_arr      = np.array(R_metal_arr)
R_total_arr      = np.array(R_total_arr)
rate_lim_arr     = np.array(rate_lim_arr)
valid            = ~np.isnan(J_arr)

# ── Rate-limiting classification array (for plot shading) ────────────────────
rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_metal_arr   > 0.5, 'metal', 'mixed'))

# ── Regime summary ───────────────────────────────────────────────────────────
# Count points in each regime
n_surface = np.sum((frac_surface_arr > 0.5)  & valid)
n_metal   = np.sum((frac_metal_arr   > 0.5)  & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_metal_arr <= 0.5) & valid
)

# Pressure ranges for each dominant regime (>90% — clean, not transitional)
surf_dom_mask  = (frac_surface_arr > 0.90) & valid
metal_dom_mask = (frac_metal_arr   > 0.90) & valid

P_surf_dom  = P_RANGE_ARR[surf_dom_mask]
P_metal_dom = P_RANGE_ARR[metal_dom_mask]

# ── Arrhenius pressure selection ─────────────────────────────────────────────
# Use median of each dominant region — no dependence on crossover
P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Crossover pressure (purely diagnostic, not used for Ea selection) ────────
P_crossover = None
valid_cross = valid & ~np.isnan(frac_surface_arr)
if np.sum(valid_cross) > 2:
    fs_v   = frac_surface_arr[valid_cross]
    P_v    = P_RANGE_ARR[valid_cross]
    sc     = np.where(np.diff(np.sign(fs_v - 0.5)))[0]
    if len(sc) > 0:
        idx = sc[0]
        # Direction check: surface fraction must drop (not rise) at crossover
        if idx + 1 < len(fs_v) and fs_v[idx] > 0.5 and fs_v[idx + 1] < 0.5:
            P_crossover = P_v[idx]

# ── Print regime summary ─────────────────────────────────────────────────────
print("=" * 60)
print(f"L1+L6 REGIME SUMMARY  (T = {T_op-273.15:.3f}°C, {np.sum(valid)} valid points)")
print("=" * 60)

print(f"\n  Regime breakdown (fraction > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Metal-limited:    {n_metal:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
if len(P_surf_dom) > 3:
    print(f"    Surface-dominant: {len(P_surf_dom):3d} points  "
          f"P = {P_surf_dom.min():.1e} – {P_surf_dom.max():.1e} Pa")
    print(f"    → P_for_surface_Ea = {P_for_surface_Ea:.2e} Pa")
else:
    print(f"    Surface-dominant: {len(P_surf_dom):3d} points  "
          f"(insufficient — surface Ea not extractable)")

if len(P_metal_dom) > 3:
    print(f"    Metal-dominant:   {len(P_metal_dom):3d} points  "
          f"P = {P_metal_dom.min():.1e} – {P_metal_dom.max():.1e} Pa")
    print(f"    → P_for_metal_Ea  = {P_for_metal_Ea:.2e} Pa")
else:
    print(f"    Metal-dominant:   {len(P_metal_dom):3d} points  "
          f"(insufficient — metal Ea not extractable)")

print(f"\n  Crossover pressure (diagnostic only)")
if P_crossover is not None:
    print(f"    P_crossover = {P_crossover:.2e} Pa")
else:
    print(f"    No crossover detected in physical pressure range")
    dominant = 'surface-limited' if n_surface > n_metal else 'metal-limited'
    print(f"    System is predominantly {dominant} across full range")

print(f"\n  θ range:  {np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  J range:  {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
print("=" * 60)

L1+L6 REGIME SUMMARY  (T = 599.850°C, 40 valid points)

  Regime breakdown (fraction > 50%)
    Surface-limited:   15 points
    Metal-limited:     25 points
    Mixed:              0 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  13 points  P = 1.0e-07 – 2.0e-03 Pa
    → P_for_surface_Ea = 1.43e-05 Pa
    Metal-dominant:    22 points  P = 2.9e-01 – 1.0e+07 Pa
    → P_for_metal_Ea  = 1.85e+03 Pa

  Crossover pressure (diagnostic only)
    P_crossover = 1.06e-02 Pa

  θ range:  0.0000 → 0.9888
  J range:  4.61e-14 → 2.94e-05 mol/m²/s


In [ ]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation, one figure per panel
#   flux        flux against pressure, coloured by the rate-limiting step
#   coverage    θ at steady state against the Langmuir isotherm
#   resistance  resistance fractions against pressure
#   limit       limit check: k_diss → ∞ recovers Sieverts (L1)
# The figures are drawn by calculations/surface_figures.py in the house style;
# this cell computes what they show.
# =============================================================================

# ── Limit check: L1+L6 at the highest k_diss of the sweep range, against Sieverts ──
J_sieverts = beta * (np.sqrt(P_RANGE_ARR) - np.sqrt(P_down))          # L1, no surface kinetics
k_diss_sweep = np.logspace(
    np.log10(CONDITIONS['k_diss_sweep_range'][0]),
    np.log10(CONDITIONS['k_diss_sweep_range'][1]),
    CONDITIONS['n_points']
)
J_high_k = []
for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_L1L6(
            P_up, P_down, L_m,
            k_diss_sweep[-1], K_eq, D_m, K_s_m
        )
        J_high_k.append(r['J_ss'])
    except Exception:
        J_high_k.append(np.nan)
J_high_k = np.array(J_high_k)

valid_parity = ~np.isnan(J_high_k) & (J_sieverts > 0) & (J_high_k > 0)
rel_errors   = (np.abs(J_high_k[valid_parity] - J_sieverts[valid_parity])
                / J_sieverts[valid_parity] * 100)
max_err, mean_err = rel_errors.max(), rel_errors.mean()
slope_net = SF.net_slope(P_RANGE_ARR[valid], J_arr[valid])

# ── Figures ──────────────────────────────────────────────────────────────────
panels = SF.two_resistance_panels(
    P_RANGE_ARR, J_arr, theta_arr, frac_surface_arr, frac_metal_arr, K_eq, P_crossover,
    J_sieverts, J_high_k, layer='metal', k_fast=k_diss_sweep[-1], T_C=T_op - 273.15,
)
save_panels(panels, "L1L6_core_validation", FIGS, module=SF.MODULE)

# ── Validation summary ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("L1+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
print(f"  Net log-log slope:       {slope_net:.4f}")
print(f"  θ range:                 "
      f"{np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  Sieverts recovery error: {max_err:.2e}%  "
      f"({'✓' if max_err < 1.0 else '⚠'})")
if P_crossover is not None:
    print(f"  Crossover pressure:      {P_crossover:.2e} Pa")
else:
    dominant = 'surface-limited' if n_surface > n_metal else 'metal-limited'
    print(f"  No crossover — system is {dominant} across full range")

In [47]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

# ── Formatting ────────────────────────────────────────────────────────────────
df_display = df.copy()

for col in df_display.columns:
    if col == 'Rate-Limiting' or col == 'Error':
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'theta' in col.lower():
            df_display[col] = df_display[col].round(6)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

# ── Summary line ──────────────────────────────────────────────────────────────
n_surface_df = (df['Rate-Limiting'] == 'SURFACE').sum()
n_metal_df   = (df['Rate-Limiting'] == 'METAL').sum()
n_mixed_df   = (df['Rate-Limiting'] == 'MIXED').sum()
n_error_df   = (df['Rate-Limiting'] == 'ERROR').sum()

print("=" * 60)
print("L1+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Metal:       {METAL_KEY}")
print(f"  T:           {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_m:         {L_m*1e3:.3f} mm")
print(f"  k_diss:      {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:        {K_eq:.3e}  Pa⁻¹")
print(f"  D_m:         {D_m:.3e}  m²/s")
print(f"  K_s_m:       {K_s_m:.3e}  mol/m³/Pa^0.5")
print(f"\n  Total rows:        {len(df)}")
print(f"  Surface-limited:   {n_surface_df}")
print(f"  Metal-limited:     {n_metal_df}")
print(f"  Mixed:             {n_mixed_df}")
if n_error_df > 0:
    print(f"  Errors:            {n_error_df}  ⚠")
print("=" * 60)

display(df_display)

L1+L6 RESULTS DATAFRAME
  Metal:       metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:           599.850°C  (873 K)
  L_m:         1.000 mm
  k_diss:      4.611e-07  mol/m²/s/Pa
  K_eq:        8.212e-04  Pa⁻¹
  D_m:         2.487e-10  m²/s
  K_s_m:       3.834e-02  mol/m³/Pa^0.5

  Total rows:        40
  Surface-limited:   15
  Metal-limited:     25
  Mixed:             0


,P_up (Pa),P_int (Pa),J_ss (mol/m²/s),θ_surface,β_metal,R_surface,R_metal,R_total,fraction_surface (%),fraction_metal (%),Rate-Limiting
0,1.0000e-07,2.3377e-11,4.6098e-14,1.3855e-07,9.5343e-09,2.1688e+13,1.0488e+08,2.1688e+13,100.00,0.00,SURFACE
1,2.2855e-07,1.2203e-10,1.0532e-13,3.1656e-07,9.5343e-09,9.4895e+12,1.0488e+08,9.4896e+12,100.00,0.00,SURFACE
2,5.2233e-07,6.3654e-10,2.4055e-13,7.2298e-07,9.5343e-09,4.1521e+12,1.0488e+08,4.1522e+12,100.00,0.00,SURFACE
3,1.1938e-06,3.3145e-09,5.4890e-13,1.6498e-06,9.5343e-09,1.8168e+12,1.0488e+08,1.8169e+12,99.99,0.01,SURFACE
4,2.7283e-06,1.7190e-08,1.2501e-12,3.7571e-06,9.5343e-09,7.9492e+11,1.0488e+08,7.9502e+11,99.99,0.01,SURFACE
5,6.2355e-06,8.8372e-08,2.8343e-12,8.5186e-06,9.5343e-09,3.4782e+11,1.0488e+08,3.4792e+11,99.97,0.03,SURFACE
6,1.4251e-05,4.4570e-07,6.3652e-12,1.9131e-05,9.5343e-09,1.5219e+11,1.0488e+08,1.5230e+11,99.93,0.07,SURFACE
7,3.2570e-05,2.1622e-06,1.4020e-11,4.2135e-05,9.5343e-09,6.6594e+10,1.0488e+08,6.6699e+10,99.84,0.16,SURFACE
8,7.4438e-05,9.7754e-06,2.9810e-11,8.9587e-05,9.5343e-09,2.9141e+10,1.0488e+08,2.9246e+10,99.64,0.36,SURFACE
9,1.7013e-04,3.9734e-05,6.0100e-11,1.8060e-04,9.5343e-09,1.2753e+10,1.0488e+08,1.2858e+10,99.18,0.82,SURFACE


### L2a+L6: Perfect Oxide + Surface Kinetics 

In [ ]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import solve_steady_state_flux_L2aL6
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS, SUGGESTED_RANGES_LEVEL5,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM      = build_simulation_config()

OXIDE_KEY = SIM['oxide_name']
OXIDE     = OXIDES[OXIDE_KEY]
SK        = OXIDE['surface_kinetics']     # oxide surface kinetics sub-dict

# ── Pressure sweep (matches widget range) ─────────────────────────────────────
P_RANGE_ARR = np.logspace(np.log10(CONDITIONS['P_range'][0]), np.log10(CONDITIONS['P_range'][1]), CONDITIONS['n_points'])
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']
N_T         = CONDITIONS['n_points']

# ── Arrhenius helper functions ────────────────────────────────────────────────
def get_k_diss_at_T(T_K):
    """k_diss at temperature T_K via Arrhenius from config."""
    return SK['k_diss_ref'] * np.exp(
        (-SK['E_diss'] / R) * (1/T_K - 1/SK['T_ref'])
    )

def get_K_eq_at_T(T_K):
    """K_eq at temperature T_K via van't Hoff from config."""
    return SK['K_eq_ref'] * np.exp(
        (-SK['H_eq'] / R) * (1/T_K - 1/SK['T_ref'])
    )

def get_D_ox_at_T(T_K):
    """D_ox at temperature T_K via Arrhenius from config."""
    return OXIDE['D_ox_ref'] * np.exp(
        (-OXIDE['E_D_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

def get_K_ox_at_T(T_K):
    """K_ox at temperature T_K via Arrhenius from config."""
    return OXIDE['K_ox_ref'] * np.exp(
        (-OXIDE['H_sol_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

# ── Known reference activation energies ──────────────────────────────────────
E_REF_SURFACE = SK['E_diss']                          # J/mol — surface dissociation
E_REF_OXIDE   = OXIDE['E_D_ox'] + OXIDE['H_sol_ox']  # J/mol — oxide permeation

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Oxide:           {OXIDE_KEY}")
print(f"  E_ref surface:   {E_REF_SURFACE/1000:.3f} kJ/mol  (E_diss)")
print(f"  E_ref oxide:     {E_REF_OXIDE/1000:.3f} kJ/mol  (E_D_ox + H_sol_ox)")
print(f"  P range:         {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:         {T_RANGE[0]} – {T_RANGE[1]} K")

In [49]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change placeholder values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op ──────────────────────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Derived ───────────────────────────────────────────────────────────────────
alpha = D_ox * K_ox / L_ox    # oxide permeance [mol/m²/s/Pa^0.5]

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L2a+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox     = {L_ox:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Surface kinetics  [config: oxide surface_kinetics sub-dict]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s  (= k_diss / K_eq)")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox     = {D_ox:.3e}  m²/s")
print(f"    K_ox     = {K_ox:.3e}  mol/m³/Pa^0.5")

print(f"\n  Derived")
print(f"    α        = {alpha:.3e}  mol/m²/s/Pa^0.5  (= D_ox × K_ox / L_ox)")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss        = {E_REF_SURFACE/1000:.3f} kJ/mol   (surface dissociation)")
print(f"    E_D_ox+H_sol  = {E_REF_OXIDE/1000:.3f} kJ/mol   (oxide permeation)")

if SK['k_diss_ref'] <= 1e-12:
    print(f"\n  ⚠  WARNING: k_diss_ref = {SK['k_diss_ref']:.0e} looks like a placeholder.")
    print(f"     Update config oxide surface_kinetics sub-dict with literature values.")

print("\n" + "=" * 60)

L2a+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox     = 1.000e-06  m
    P_down   = 0.0e+00  Pa

  Surface kinetics  [config: oxide surface_kinetics sub-dict]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹
    k_recomb = 8.468e-05  mol/m²/s  (= k_diss / K_eq)

  Oxide transport  [config: Arrhenius]
    D_ox     = 1.395e-17  m²/s
    K_ox     = 2.869e+02  mol/m³/Pa^0.5

  Derived
    α        = 4.002e-09  mol/m²/s/Pa^0.5  (= D_ox × K_ox / L_ox)

  Reference activation energies  [config]
    E_diss        = 57.950 kJ/mol   (surface dissociation)
    E_D_ox+H_sol  = 234.000 kJ/mol   (oxide permeation)



In [50]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop ──────────────────────────────────────────────────────────
rows             = []
J_arr            = []
theta_arr        = []
P_surf_arr       = []
frac_surface_arr = []
frac_oxide_arr   = []
R_surface_arr    = []
R_oxide_arr      = []
R_total_arr      = []
rate_lim_arr     = []

for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_L2aL6(
            P_up, P_down,
            k_diss, K_eq, D_ox, K_ox, L_ox
        )
        J_arr.append(r['J_ss'])
        theta_arr.append(r['theta'])
        P_surf_arr.append(r['P_surf'])
        frac_surface_arr.append(r['resistances']['fraction_surface'])
        frac_oxide_arr.append(r['resistances']['fraction_oxide'])
        R_surface_arr.append(r['resistances']['R_surface'])
        R_oxide_arr.append(r['resistances']['R_oxide'])
        R_total_arr.append(r['resistances']['R_total'])
        rate_lim_arr.append(r['rate_limiting'])

        rows.append({
            'P_up (Pa)':            P_up,
            'P_surf (Pa)':          r['P_surf'],
            'J_ss (mol/m²/s)':      r['J_ss'],
            'θ_surface':            r['theta'],
            'α_oxide':              r['alpha'],
            'R_surface':            r['resistances']['R_surface'],
            'R_oxide':              r['resistances']['R_oxide'],
            'R_total':              r['resistances']['R_total'],
            'fraction_surface (%)': r['resistances']['fraction_surface'] * 100,
            'fraction_oxide (%)':   r['resistances']['fraction_oxide']   * 100,
            'Rate-Limiting':        r['rate_limiting'].upper(),
        })

    except Exception as e:
        J_arr.append(np.nan);            theta_arr.append(np.nan)
        P_surf_arr.append(np.nan);       frac_surface_arr.append(np.nan)
        frac_oxide_arr.append(np.nan);   R_surface_arr.append(np.nan)
        R_oxide_arr.append(np.nan);      R_total_arr.append(np.nan)
        rate_lim_arr.append('error')
        rows.append({'P_up (Pa)': P_up, 'Rate-Limiting': 'ERROR', 'Error': str(e)})

# ── Convert to arrays ─────────────────────────────────────────────────────────
J_arr            = np.array(J_arr)
theta_arr        = np.array(theta_arr)
P_surf_arr       = np.array(P_surf_arr)
frac_surface_arr = np.array(frac_surface_arr)
frac_oxide_arr   = np.array(frac_oxide_arr)
R_surface_arr    = np.array(R_surface_arr)
R_oxide_arr      = np.array(R_oxide_arr)
R_total_arr      = np.array(R_total_arr)
rate_lim_arr     = np.array(rate_lim_arr)
valid            = ~np.isnan(J_arr)

# ── Rate-limiting classification array ───────────────────────────────────────
rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_oxide_arr   > 0.5, 'oxide', 'mixed'))

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) & valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]

# ── Arrhenius pressure selection (no crossover dependence) ───────────────────
P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None

# ── Crossover pressure (diagnostic only) ─────────────────────────────────────
P_crossover = None
valid_cross = valid & ~np.isnan(frac_surface_arr)
if np.sum(valid_cross) > 2:
    fs_v = frac_surface_arr[valid_cross]
    P_v  = P_RANGE_ARR[valid_cross]
    sc   = np.where(np.diff(np.sign(fs_v - 0.5)))[0]
    if len(sc) > 0:
        idx = sc[0]
        if idx + 1 < len(fs_v) and fs_v[idx] > 0.5 and fs_v[idx+1] < 0.5:
            P_crossover = P_v[idx]

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L2a+L6 REGIME SUMMARY  (T = {T_op-273.15:.3f}°C, {np.sum(valid)} valid points)")
print("=" * 60)

print(f"\n  Regime breakdown (fraction > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Oxide-limited:    {n_oxide:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
if len(P_surf_dom) > 3:
    print(f"    Surface-dominant: {len(P_surf_dom):3d} points  "
          f"P = {P_surf_dom.min():.1e} – {P_surf_dom.max():.1e} Pa")
    print(f"    → P_for_surface_Ea = {P_for_surface_Ea:.2e} Pa")
else:
    print(f"    Surface-dominant: {len(P_surf_dom):3d} points  "
          f"(insufficient — surface Ea not extractable)")

if len(P_oxide_dom) > 3:
    print(f"    Oxide-dominant:   {len(P_oxide_dom):3d} points  "
          f"P = {P_oxide_dom.min():.1e} – {P_oxide_dom.max():.1e} Pa")
    print(f"    → P_for_oxide_Ea  = {P_for_oxide_Ea:.2e} Pa")
else:
    print(f"    Oxide-dominant:   {len(P_oxide_dom):3d} points  "
          f"(insufficient — oxide Ea not extractable)")

print(f"\n  Crossover pressure (diagnostic only)")
if P_crossover is not None:
    print(f"    P_crossover = {P_crossover:.2e} Pa")
else:
    print(f"    No crossover detected in physical pressure range")
    dominant = 'surface-limited' if n_surface > n_oxide else 'oxide-limited'
    print(f"    System is predominantly {dominant} across full range")

print(f"\n  θ range:  {np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  J range:  {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
print("=" * 60)

L2a+L6 REGIME SUMMARY  (T = 599.850°C, 40 valid points)

  Regime breakdown (fraction > 50%)
    Surface-limited:   21 points
    Oxide-limited:     19 points
    Mixed:              0 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  18 points  P = 1.0e-07 – 1.3e-01 Pa
    → P_for_surface_Ea = 1.22e-04 Pa
    Oxide-dominant:    17 points  P = 1.8e+01 – 1.0e+07 Pa
    → P_for_oxide_Ea  = 1.34e+04 Pa

  Crossover pressure (diagnostic only)
    P_crossover = 1.51e+00 Pa

  θ range:  0.0000 → 0.9396
  J range:  2.37e-16 → 1.18e-05 mol/m²/s


In [ ]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation, one figure per panel
#   flux        flux against pressure, coloured by the rate-limiting step
#   coverage    θ at steady state against the Langmuir isotherm
#   resistance  resistance fractions against pressure (surface and oxide sum to 100 %)
#   limit       limit check: k_diss → ∞ recovers L2a
# The figures are drawn by calculations/surface_figures.py in the house style;
# this cell computes what they show.
# =============================================================================

# ── Limit check: L2a+L6 at fast kinetics (k_diss = 1e-3), against L2a ────────
k_fast = 1e-3
J_L2a  = alpha * (np.sqrt(P_RANGE_ARR) - np.sqrt(P_down))             # L2a, no surface kinetics
J_high_k = []
for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_L2aL6(
            P_up, P_down,
            k_fast, K_eq, D_ox, K_ox, L_ox    # k_diss → large
        )
        J_high_k.append(r['J_ss'])
    except Exception:
        J_high_k.append(np.nan)
J_high_k = np.array(J_high_k)

valid_parity = ~np.isnan(J_high_k) & (J_L2a > 0) & (J_high_k > 0)
rel_errors   = (np.abs(J_high_k[valid_parity] - J_L2a[valid_parity])
                / J_L2a[valid_parity] * 100)
max_err, mean_err = rel_errors.max(), rel_errors.mean()
slope_net = SF.net_slope(P_RANGE_ARR[valid], J_arr[valid])

# ── Figures ──────────────────────────────────────────────────────────────────
panels = SF.two_resistance_panels(
    P_RANGE_ARR, J_arr, theta_arr, frac_surface_arr, frac_oxide_arr, K_eq, P_crossover,
    J_L2a, J_high_k, layer='oxide', k_fast=k_fast, T_C=T_op - 273.15,
)
save_panels(panels, "L2aL6_core_validation", FIGS, module=SF.MODULE)

print("\n" + "=" * 60)
print("L2a+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
print(f"  Net log-log slope:         {slope_net:.4f}")
print(f"  θ range:                   "
      f"{np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  L2a recovery error:        {max_err:.2e}%  "
      f"({'✓' if max_err < 1.0 else '⚠'})")
if P_crossover is not None:
    print(f"  Crossover pressure:        {P_crossover:.2e} Pa")
else:
    dominant = 'surface-limited' if n_surface > n_oxide else 'oxide-limited'
    print(f"  No crossover — system is {dominant} across full range")
print("=" * 60)

In [53]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('Rate-Limiting', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'theta' in col.lower():
            df_display[col] = df_display[col].round(6)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['Rate-Limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['Rate-Limiting'] == 'OXIDE').sum()
n_mixed_df   = (df['Rate-Limiting'] == 'MIXED').sum()
n_error_df   = (df['Rate-Limiting'] == 'ERROR').sum()

print("=" * 60)
print("L2a+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:       {OXIDE_KEY}")
print(f"  T:           {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:        {L_ox*1e6:.3f} μm")
print(f"  k_diss:      {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:        {K_eq:.3e}  Pa⁻¹")
print(f"  D_ox:        {D_ox:.3e}  m²/s")
print(f"  K_ox:        {K_ox:.3e}  mol/m³/Pa^0.5")
print(f"\n  Total rows:        {len(df)}")
print(f"  Surface-limited:   {n_surface_df}")
print(f"  Oxide-limited:     {n_oxide_df}")
print(f"  Mixed:             {n_mixed_df}")
if n_error_df > 0:
    print(f"  Errors:            {n_error_df}  ⚠")
print("=" * 60)

display(df_display)

L2a+L6 RESULTS DATAFRAME
  Oxide:       Cr2O3_sample4
  T:           599.850°C  (873 K)
  L_ox:        1.000 μm
  k_diss:      2.370e-09  mol/m²/s/Pa
  K_eq:        2.799e-05  Pa⁻¹
  D_ox:        1.395e-17  m²/s
  K_ox:        2.869e+02  mol/m³/Pa^0.5

  Total rows:        40
  Surface-limited:   21
  Oxide-limited:     19
  Mixed:             0


,P_up (Pa),P_surf (Pa),J_ss (mol/m²/s),θ_surface,α_oxide,R_surface,R_oxide,R_total,fraction_surface (%),fraction_oxide (%),Rate-Limiting
0,1.0000e-07,3.5082e-15,2.3702e-16,3.1336e-10,4.0017e-09,4.2191e+15,2.4990e+08,4.2191e+15,100.00,0.00,SURFACE
1,2.2855e-07,1.8325e-14,5.4170e-16,7.1616e-10,4.0017e-09,1.8460e+15,2.4990e+08,1.8460e+15,100.00,0.00,SURFACE
2,5.2233e-07,9.5716e-14,1.2380e-15,1.6368e-09,4.0017e-09,8.0773e+14,2.4990e+08,8.0773e+14,100.00,0.00,SURFACE
3,1.1938e-06,4.9996e-13,2.8295e-15,3.7408e-09,4.0017e-09,3.5342e+14,2.4990e+08,3.5342e+14,100.00,0.00,SURFACE
4,2.7283e-06,2.6114e-12,6.4667e-15,8.5494e-09,4.0017e-09,1.5464e+14,2.4990e+08,1.5464e+14,100.00,0.00,SURFACE
5,6.2355e-06,1.3640e-11,1.4779e-14,1.9539e-08,4.0017e-09,6.7662e+13,2.4990e+08,6.7662e+13,100.00,0.00,SURFACE
6,1.4251e-05,7.1249e-11,3.3778e-14,4.4656e-08,4.0017e-09,2.9605e+13,2.4990e+08,2.9606e+13,100.00,0.00,SURFACE
7,3.2570e-05,3.7215e-10,7.7197e-14,1.0206e-07,4.0017e-09,1.2954e+13,2.4990e+08,1.2954e+13,100.00,0.00,SURFACE
8,7.4438e-05,1.9438e-09,1.7643e-13,2.3325e-07,4.0017e-09,5.6679e+12,2.4990e+08,5.6681e+12,100.00,0.00,SURFACE
9,1.7013e-04,1.0152e-08,4.0321e-13,5.3307e-07,4.0017e-09,2.4800e+12,2.4990e+08,2.4802e+12,99.99,0.01,SURFACE


### L2a +L6

In [ ]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import solve_steady_state_flux_direct
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM       = build_simulation_config()

METAL_KEY = SIM['metal_name']
OXIDE_KEY = SIM['oxide_name']
METAL     = METALS[METAL_KEY]
OXIDE     = OXIDES[OXIDE_KEY]
SK        = OXIDE['surface_kinetics']    # oxide surface kinetics sub-dict

# ── Pressure sweep ────────────────────────────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']

In [55]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change placeholder values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
L_m    = SIM['L_metal']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op ──────────────────────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Metal transport at T_op ───────────────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Derived ───────────────────────────────────────────────────────────────────
alpha = D_ox * K_ox / L_ox
beta  = D_m  * K_s_m / L_m

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L2+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox     = {L_ox:.3e}  m")
print(f"    L_m      = {L_m:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Surface kinetics  [config: oxide surface_kinetics sub-dict]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s  (= k_diss / K_eq)")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox     = {D_ox:.3e}  m²/s")
print(f"    K_ox     = {K_ox:.3e}  mol/m³/Pa^0.5")

print(f"\n  Metal transport  [config: Arrhenius]")
print(f"    D_m      = {D_m:.3e}  m²/s")
print(f"    K_s_m    = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Derived")
print(f"    α        = {alpha:.3e}  mol/m²/s/Pa^0.5  (= D_ox × K_ox / L_ox)")
print(f"    β        = {beta:.3e}  mol/m²/s/Pa^0.5  (= D_m × K_s_m / L_m)")
print(f"    α/β      = {alpha/beta:.3e}  (oxide/metal permeance ratio)")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss        = {E_REF_SURFACE/1000:.3f} kJ/mol   (surface)")
print(f"    E_D_ox+H_sol  = {E_REF_OXIDE/1000:.3f} kJ/mol   (oxide permeation)")
print(f"    E_D+H_s       = {E_REF_METAL/1000:.3f} kJ/mol   (metal permeation)")

if SK['k_diss_ref'] <= 1e-12:
    print(f"\n  ⚠  WARNING: k_diss_ref = {SK['k_diss_ref']:.0e} looks like a placeholder.")
    print(f"     Update config oxide surface_kinetics sub-dict with literature values.")

print("\n" + "=" * 60)

L2+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox     = 1.000e-06  m
    L_m      = 1.000e-03  m
    P_down   = 0.0e+00  Pa

  Surface kinetics  [config: oxide surface_kinetics sub-dict]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹
    k_recomb = 8.468e-05  mol/m²/s  (= k_diss / K_eq)

  Oxide transport  [config: Arrhenius]
    D_ox     = 1.395e-17  m²/s
    K_ox     = 2.869e+02  mol/m³/Pa^0.5

  Metal transport  [config: Arrhenius]
    D_m      = 2.487e-10  m²/s
    K_s_m    = 3.834e-02  mol/m³/Pa^0.5

  Derived
    α        = 4.002e-09  mol/m²/s/Pa^0.5  (= D_ox × K_ox / L_ox)
    β        = 9.534e-09  mol/m²/s/Pa^0.5  (= D_m × K_s_m / L_m)
    α/β      = 4.197e-01  (oxide/metal permeance ratio)

  Reference activation energies  [config]
    E_diss        = 57.950 kJ/mol   (surface)
    E_D_ox+H_sol  = 234.000 kJ/mol   (oxide permeation)
    E_D+H_s       = 79.400 kJ/mol   (metal permeation)



In [56]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop ──────────────────────────────────────────────────────────
rows             = []
J_arr            = []
theta_arr        = []
P_int_arr        = []
frac_surface_arr = []
frac_oxide_arr   = []
frac_metal_arr   = []
R_surface_arr    = []
R_oxide_arr      = []
R_metal_arr      = []
R_total_arr      = []
rate_lim_arr     = []

for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m
        )
        J_arr.append(r['J_ss'])
        theta_arr.append(r['theta'])
        P_int_arr.append(r['P_int'])
        frac_surface_arr.append(r['resistances']['fraction_surface'])
        frac_oxide_arr.append(r['resistances']['fraction_oxide'])
        frac_metal_arr.append(r['resistances']['fraction_metal'])
        R_surface_arr.append(r['resistances']['R_surface'])
        R_oxide_arr.append(r['resistances']['R_oxide'])
        R_metal_arr.append(r['resistances']['R_metal'])
        R_total_arr.append(r['resistances']['R_total'])
        rate_lim_arr.append(r['rate_limiting'])

        rows.append({
            'P_up (Pa)':            P_up,
            'P_int (Pa)':           r['P_int'],
            'J_ss (mol/m²/s)':      r['J_ss'],
            'θ_surface':            r['theta'],
            'α_oxide':              r['alpha'],
            'β_metal':              r['beta'],
            'R_surface':            r['resistances']['R_surface'],
            'R_oxide':              r['resistances']['R_oxide'],
            'R_metal':              r['resistances']['R_metal'],
            'R_total':              r['resistances']['R_total'],
            'fraction_surface (%)': r['resistances']['fraction_surface'] * 100,
            'fraction_oxide (%)':   r['resistances']['fraction_oxide']   * 100,
            'fraction_metal (%)':   r['resistances']['fraction_metal']   * 100,
            'Rate-Limiting':        r['rate_limiting'].upper(),
        })

    except Exception as e:
        for lst in [J_arr, theta_arr, P_int_arr, frac_surface_arr,
                    frac_oxide_arr, frac_metal_arr, R_surface_arr,
                    R_oxide_arr, R_metal_arr, R_total_arr]:
            lst.append(np.nan)
        rate_lim_arr.append('error')
        rows.append({'P_up (Pa)': P_up, 'Rate-Limiting': 'ERROR', 'Error': str(e)})

# ── Convert to arrays ─────────────────────────────────────────────────────────
J_arr            = np.array(J_arr)
theta_arr        = np.array(theta_arr)
P_int_arr        = np.array(P_int_arr)
frac_surface_arr = np.array(frac_surface_arr)
frac_oxide_arr   = np.array(frac_oxide_arr)
frac_metal_arr   = np.array(frac_metal_arr)
R_surface_arr    = np.array(R_surface_arr)
R_oxide_arr      = np.array(R_oxide_arr)
R_metal_arr      = np.array(R_metal_arr)
R_total_arr      = np.array(R_total_arr)
rate_lim_arr     = np.array(rate_lim_arr)
valid            = ~np.isnan(J_arr)

# ── Rate-limiting classification array ───────────────────────────────────────
rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_oxide_arr   > 0.5, 'oxide',
         np.where(frac_metal_arr   > 0.5, 'metal', 'mixed')))

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & valid)
n_metal   = np.sum((frac_metal_arr   > 0.5) & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) &
    (frac_metal_arr   <= 0.5) & valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & valid
metal_dom_mask = (frac_metal_arr   > 0.90) & valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]
P_metal_dom    = P_RANGE_ARR[metal_dom_mask]

# ── Arrhenius pressure selection (no crossover dependence) ───────────────────
P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Crossover pressures (diagnostic only) ────────────────────────────────────
def find_crossover(frac_a, frac_b, valid_mask):
    """Find pressure where frac_a crosses frac_b (from above)."""
    vc = valid_mask & ~np.isnan(frac_a) & ~np.isnan(frac_b)
    if np.sum(vc) < 2:
        return None
    diff = frac_a[vc] - frac_b[vc]
    Pv   = P_RANGE_ARR[vc]
    sc   = np.where(np.diff(np.sign(diff)))[0]
    if len(sc) == 0:
        return None
    idx = sc[0]
    if diff[idx] > 0 and diff[idx+1] < 0:
        return Pv[idx]
    return None

P_cross_surf_oxide = find_crossover(frac_surface_arr, frac_oxide_arr, valid)
P_cross_surf_metal = find_crossover(frac_surface_arr, frac_metal_arr, valid)
P_cross_oxide_metal= find_crossover(frac_oxide_arr,   frac_metal_arr, valid)

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L2+L6 REGIME SUMMARY  (T = {T_op-273.15:.3f}°C, {np.sum(valid)} valid points)")
print("=" * 60)

print(f"\n  Regime breakdown (dominant fraction > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Oxide-limited:    {n_oxide:3d} points")
print(f"    Metal-limited:    {n_metal:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
for label, dom, P_ea in [
    ('Surface', P_surf_dom,  P_for_surface_Ea),
    ('Oxide',   P_oxide_dom, P_for_oxide_Ea),
    ('Metal',   P_metal_dom, P_for_metal_Ea),
]:
    if len(dom) > 3:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"P = {dom.min():.1e} – {dom.max():.1e} Pa  "
              f"→ P_for_{label.lower()}_Ea = {P_ea:.2e} Pa")
    else:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"(insufficient — {label.lower()} Ea not extractable)")

print(f"\n  Crossover pressures (diagnostic only)")
for desc, P_co in [
    ('Surface → Oxide',  P_cross_surf_oxide),
    ('Surface → Metal',  P_cross_surf_metal),
    ('Oxide   → Metal',  P_cross_oxide_metal),
]:
    val = f'{P_co:.2e} Pa' if P_co is not None else 'not found'
    print(f"    {desc}:  {val}")

print(f"\n  α/β ratio:  {alpha/beta:.3e}")
print(f"  θ range:    {np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  J range:    {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
print("=" * 60)

L2+L6 REGIME SUMMARY  (T = 599.850°C, 40 valid points)

  Regime breakdown (dominant fraction > 50%)
    Surface-limited:   20 points
    Oxide-limited:     19 points
    Metal-limited:      0 points
    Mixed:              1 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  18 points  P = 1.0e-07 – 1.3e-01 Pa  → P_for_surface_Ea = 1.22e-04 Pa
    Oxide-dominant:   0 points  (insufficient — oxide Ea not extractable)
    Metal-dominant:   0 points  (insufficient — metal Ea not extractable)

  Crossover pressures (diagnostic only)
    Surface → Oxide:  1.51e+00 Pa
    Surface → Metal:  3.46e+00 Pa
    Oxide   → Metal:  not found

  α/β ratio:  4.197e-01
  θ range:    0.0000 → 0.9407
  J range:    2.37e-16 → 8.45e-06 mol/m²/s


In [ ]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation, one figure per panel
#   flux        flux against pressure, coloured by the rate-limiting step (three resistances)
#   arrhenius   apparent E_a per regime: flux against 1000/T at one pressure per regime
#   resistance  resistance fractions against pressure, with the S→O and O→M crossovers
#   limit       limit check: k_diss → ∞ recovers the series-resistance flux (no L6)
# The figures are drawn by calculations/surface_figures.py in the house style;
# this cell computes what they show.
# =============================================================================

# ── Arrhenius: apparent E_a per regime, classified by regime purity ──────────
# Sweeps temperature at up to three fixed pressures, one per dominant regime (the median of the
# range where that step holds over 90 % of the resistance, from Cell 3). A linear fit of ln(J)
# against 1000/T through the points where that step holds 99 % gives the apparent E_a, compared
# with the configured references: E_diss (surface), E_D_ox+H_sol_ox (oxide), E_D+H_s (metal).
temperatures_K = np.linspace(T_RANGE[0], T_RANGE[1], N_T)
inv_T          = 1000.0 / temperatures_K
arrhenius_results = {}

# pure_thresh is 0.99, not 0.90: a point that is 90% surface-controlled still
# carries 10% of a path whose activation energy is ~234 kJ/mol against the
# surface's ~58, so admitting it biases the extracted E_diss upward by ~10
# kJ/mol. Measured: at 0.90 the fit returns 67.6, at 0.99 it returns 59.2
# against a configured 57.95, and the oxide fit moves 217 -> 231 against 234.
def classify_regime(fs, fo, fm, pure_thresh=0.99, mixed_thresh=0.50):
    """
    Classify a single T point into one of seven regime categories
    based on resistance fractions at that temperature and pressure.
    pure_thresh  : fraction above which a step is called 'pure'
    mixed_thresh : fraction above which a step is called 'dominant'
    """
    if fs >= pure_thresh:    return 'pure_surface'
    elif fo >= pure_thresh:  return 'pure_oxide'
    elif fm >= pure_thresh:  return 'pure_metal'
    elif fs >= mixed_thresh: return 'mixed_surface'
    elif fo >= mixed_thresh: return 'mixed_oxide'
    elif fm >= mixed_thresh: return 'mixed_metal'
    else:                    return 'mixed'

E_refs    = {'surface': E_REF_SURFACE, 'oxide': E_REF_OXIDE, 'metal': E_REF_METAL}
regime_Ea_cases = [
    ('surface', P_for_surface_Ea),
    ('oxide',   P_for_oxide_Ea),
    ('metal',   P_for_metal_Ea),
]
arrhenius_cases = []

for label, P_fixed in regime_Ea_cases:
    if P_fixed is None:
        arrhenius_results[label] = None
        arrhenius_cases.append({'regime': label, 'P_fixed': None, 'J': None, 'E_ref': E_refs[label]})
        continue

    J_row, class_row = [], []
    for T_K in temperatures_K:
        try:
            r = solve_steady_state_flux_direct(
                P_fixed, P_down, L_m,
                get_k_diss_at_T(T_K), get_K_eq_at_T(T_K),
                get_D_ox_at_T(T_K),   get_K_ox_at_T(T_K), L_ox,
                get_D_m_at_T(T_K),    get_K_s_m_at_T(T_K),
            )
            J_row.append(r['J_ss'])
            fs = r['resistances']['fraction_surface']
            fo = r['resistances']['fraction_oxide']
            fm = r['resistances']['fraction_metal']
            class_row.append(classify_regime(fs, fo, fm))
        except Exception:
            J_row.append(np.nan)
            class_row.append('mixed')

    J_arr_T   = np.array(J_row)
    class_arr = np.array(class_row)

    # Pure mask — only points classified as pure target regime
    pure = (class_arr == f'pure_{label}') & ~np.isnan(J_arr_T) & (J_arr_T > 0)

    # Arrhenius fit on pure points only
    if np.sum(pure) < 3:
        arrhenius_results[label] = None
    else:
        slope, intercept, r_val, *_ = stats.linregress(inv_T[pure], np.log(J_arr_T[pure]))
        arrhenius_results[label] = {
            'J':            J_arr_T,
            'valid':        pure,
            'slope':        slope,
            'intercept':    intercept,
            'r_sq':         r_val**2,
            'E_a':          -slope * R * 1000,
            'P_fixed':      P_fixed,
            'n_pure':       int(np.sum(pure)),
            'T_pure_range': (temperatures_K[pure].min() - 273.15,
                             temperatures_K[pure].max() - 273.15),
            'class_arr':    class_arr,
        }
    arrhenius_cases.append({'regime': label, 'P_fixed': P_fixed, 'J': J_arr_T, 'classes': class_arr,
                            'fit': arrhenius_results[label], 'E_ref': E_refs[label]})

# ── Limit check: L2+L6 at fast kinetics (k_diss = 1e-3), against series resistance ──
# J = (sqrt(P_up) - sqrt(P_down)) / (1/alpha + 1/beta)
k_fast   = 1e-3
J_series = ((np.sqrt(P_RANGE_ARR) - np.sqrt(P_down)) /
            (1.0/alpha + 1.0/beta))
J_high_k = []
for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            k_fast, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m
        )
        J_high_k.append(r['J_ss'])
    except Exception:
        J_high_k.append(np.nan)
J_high_k = np.array(J_high_k)

valid_parity = ~np.isnan(J_high_k) & (J_series > 0) & (J_high_k > 0)
rel_errors   = (np.abs(J_high_k[valid_parity] - J_series[valid_parity])
                / J_series[valid_parity] * 100)
max_err, mean_err = rel_errors.max(), rel_errors.mean()
slope_net = SF.net_slope(P_RANGE_ARR[valid], J_arr[valid])

# ── Figures ──────────────────────────────────────────────────────────────────
panels = SF.l2l6_panels(
    P_RANGE_ARR, J_arr, frac_surface_arr, frac_oxide_arr, frac_metal_arr,
    [('S→O', P_cross_surf_oxide), ('O→M', P_cross_oxide_metal)],
    {'inv_T': inv_T, 'cases': arrhenius_cases}, J_series, J_high_k, k_fast=k_fast, T_C=T_op - 273.15,
)
save_panels(panels, "L2L6_core_validation", FIGS, module=SF.MODULE)

print("\n" + "=" * 60)
print("L2+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
print(f"  Net log-log slope:       {slope_net:.4f}")
print(f"  θ range:                 "
      f"{np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  Series limit error:      {max_err:.2e}%  "
      f"({'✓' if max_err < 1.0 else '⚠'})")
print(f"\n  Arrhenius results:")
for lbl in ('surface', 'oxide', 'metal'):
    res = arrhenius_results.get(lbl)
    E_r = E_refs[lbl]
    if res is None:
        print(f"    {lbl.capitalize()}: not extractable")
    else:
        match = abs(res['E_a'] - E_r) < 3000
        print(f"    {lbl.capitalize()}: extracted={res['E_a']/1000:.3f} "
              f"expected={E_r/1000:.3f} kJ/mol  "
              f"{'✓' if match else '⚠'}")
print("=" * 60)

In [59]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('Rate-Limiting', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'theta' in col.lower():
            df_display[col] = df_display[col].round(6)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['Rate-Limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['Rate-Limiting'] == 'OXIDE').sum()
n_metal_df   = (df['Rate-Limiting'] == 'METAL').sum()
n_mixed_df   = (df['Rate-Limiting'] == 'MIXED').sum()
n_error_df   = (df['Rate-Limiting'] == 'ERROR').sum()

print("=" * 60)
print("L2+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:       {OXIDE_KEY}")
print(f"  Metal:       {METAL_KEY}")
print(f"  T:           {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:        {L_ox*1e6:.3f} μm")
print(f"  L_m:         {L_m*1e3:.3f} mm")
print(f"  k_diss:      {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:        {K_eq:.3e}  Pa⁻¹")
print(f"  α:           {alpha:.3e}  mol/m²/s/Pa^0.5")
print(f"  β:           {beta:.3e}  mol/m²/s/Pa^0.5")
print(f"  α/β:         {alpha/beta:.3e}")
print(f"\n  Total rows:        {len(df)}")
print(f"  Surface-limited:   {n_surface_df}")
print(f"  Oxide-limited:     {n_oxide_df}")
print(f"  Metal-limited:     {n_metal_df}")
print(f"  Mixed:             {n_mixed_df}")
if n_error_df > 0:
    print(f"  Errors:            {n_error_df}  ⚠")
print("=" * 60)

display(df_display)

L2+L6 RESULTS DATAFRAME
  Oxide:       Cr2O3_sample4
  Metal:       metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:           599.850°C  (873 K)
  L_ox:        1.000 μm
  L_m:         1.000 mm
  k_diss:      2.370e-09  mol/m²/s/Pa
  K_eq:        2.799e-05  Pa⁻¹
  α:           4.002e-09  mol/m²/s/Pa^0.5
  β:           9.534e-09  mol/m²/s/Pa^0.5
  α/β:         4.197e-01

  Total rows:        40
  Surface-limited:   20
  Oxide-limited:     19
  Metal-limited:     0
  Mixed:             1


,P_up (Pa),P_int (Pa),J_ss (mol/m²/s),θ_surface,α_oxide,β_metal,R_surface,R_oxide,R_metal,R_total,fraction_surface (%),fraction_oxide (%),fraction_metal (%),Rate-Limiting
0,1.0000e-07,6.1800e-16,2.3702e-16,4.4487e-10,4.0017e-09,9.5343e-09,4.2191e+15,2.4990e+08,1.0488e+08,4.2191e+15,100.00,0.00,0.00,SURFACE
1,2.2855e-07,3.2280e-15,5.4170e-16,1.0167e-09,4.0017e-09,9.5343e-09,1.8460e+15,2.4990e+08,1.0488e+08,1.8460e+15,100.00,0.00,0.00,SURFACE
2,5.2233e-07,1.6861e-14,1.2380e-15,2.3237e-09,4.0017e-09,9.5343e-09,8.0773e+14,2.4990e+08,1.0488e+08,8.0773e+14,100.00,0.00,0.00,SURFACE
3,1.1938e-06,8.8071e-14,2.8295e-15,5.3108e-09,4.0017e-09,9.5343e-09,3.5342e+14,2.4990e+08,1.0488e+08,3.5342e+14,100.00,0.00,0.00,SURFACE
4,2.7283e-06,4.6002e-13,6.4667e-15,1.2138e-08,4.0017e-09,9.5343e-09,1.5464e+14,2.4990e+08,1.0488e+08,1.5464e+14,100.00,0.00,0.00,SURFACE
5,6.2355e-06,2.4029e-12,1.4779e-14,2.7740e-08,4.0017e-09,9.5343e-09,6.7662e+13,2.4990e+08,1.0488e+08,6.7662e+13,100.00,0.00,0.00,SURFACE
6,1.4251e-05,1.2551e-11,3.3777e-14,6.3398e-08,4.0017e-09,9.5343e-09,2.9605e+13,2.4990e+08,1.0488e+08,2.9606e+13,100.00,0.00,0.00,SURFACE
7,3.2570e-05,6.5555e-11,7.7196e-14,1.4489e-07,4.0017e-09,9.5343e-09,1.2954e+13,2.4990e+08,1.0488e+08,1.2954e+13,100.00,0.00,0.00,SURFACE
8,7.4438e-05,3.4240e-10,1.7642e-13,3.3114e-07,4.0017e-09,9.5343e-09,5.6679e+12,2.4990e+08,1.0488e+08,5.6683e+12,99.99,0.00,0.00,SURFACE
9,1.7013e-04,1.7882e-09,4.0318e-13,7.5675e-07,4.0017e-09,9.5343e-09,2.4800e+12,2.4990e+08,1.0488e+08,2.4803e+12,99.99,0.01,0.00,SURFACE


## Level 3+ L6: Surface Kinetics with Mixed Regimes

In [ ]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import (
    calculate_mixed_defect_flux_L6,
    calculate_path_flux_L6,
    solve_steady_state_flux_direct,
)
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS, SUGGESTED_RANGES_LEVEL5,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM       = build_simulation_config()

METAL_KEY  = SIM['metal_name']
OXIDE_KEY  = SIM['oxide_name']
METAL      = METALS[METAL_KEY]
OXIDE      = OXIDES[OXIDE_KEY]
SK_OX      = OXIDE['surface_kinetics']     # oxide surface kinetics
SK_M       = METAL['surface_kinetics']     # metal surface kinetics (pinhole)

# ── Pressure sweep ────────────────────────────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']
N_T         = CONDITIONS['n_points']

# ── Arrhenius helper functions — oxide ───────────────────────────────────────
def get_k_diss_at_T(T_K):
    return SK_OX['k_diss_ref'] * np.exp(
        (-SK_OX['E_diss'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_K_eq_at_T(T_K):
    return SK_OX['K_eq_ref'] * np.exp(
        (-SK_OX['H_eq'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_D_ox_at_T(T_K):
    return OXIDE['D_ox_ref'] * np.exp(
        (-OXIDE['E_D_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

def get_K_ox_at_T(T_K):
    return OXIDE['K_ox_ref'] * np.exp(
        (-OXIDE['H_sol_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

# ── Arrhenius helper functions — metal ───────────────────────────────────────
def get_D_m_at_T(T_K):
    return METAL['D_ref'] * np.exp(
        (-METAL['E_D'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_K_s_m_at_T(T_K):
    return METAL['K_s_ref'] * np.exp(
        (-METAL['H_s'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_k_diss_metal_at_T(T_K):
    """Metal surface k_diss — used for pinhole paths when not Sieverts limit."""
    return SK_M['k_diss_metal_ref'] * np.exp(
        (-SK_M['E_diss_metal'] / R) * (1/T_K - 1/SK_M['T_ref'])
    )

def get_K_eq_metal_at_T(T_K):
    """Metal surface K_eq — used for pinhole paths when not Sieverts limit."""
    return SK_M['K_eq_metal_ref'] * np.exp(
        (-SK_M['H_eq_metal'] / R) * (1/T_K - 1/SK_M['T_ref'])
    )

# ── Known reference activation energies ──────────────────────────────────────
E_REF_SURFACE = SK_OX['E_diss']
E_REF_OXIDE   = OXIDE['E_D_ox'] + OXIDE['H_sol_ox']
E_REF_METAL   = METAL['E_D']    + METAL['H_s']

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Metal:           {METAL_KEY}")
print(f"  Oxide:           {OXIDE_KEY}")
print(f"  E_ref surface:   {E_REF_SURFACE/1000:.3f} kJ/mol  (E_diss)")
print(f"  E_ref oxide:     {E_REF_OXIDE/1000:.3f} kJ/mol  (E_D_ox + H_sol_ox)")
print(f"  E_ref metal:     {E_REF_METAL/1000:.3f} kJ/mol  (E_D + H_s)")
print(f"  P range:         {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:         {T_RANGE[0]} – {T_RANGE[1]} K")

In [61]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
L_m    = SIM['L_metal']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op — oxide surface ──────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Metal transport at T_op ───────────────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Metal surface kinetics at T_op — for pinhole paths ───────────────────────
k_diss_metal = get_k_diss_metal_at_T(T_op)
K_eq_metal   = get_K_eq_metal_at_T(T_op)

# ── Derived permeances ────────────────────────────────────────────────────────
alpha_intact = D_ox * K_ox / L_ox
beta         = D_m  * K_s_m / L_m

# ── Defect configuration — from SIM (built by build_simulation_config) ────────
defect_config       = SIM['defect_config']
use_sieverts_pinhole = SIM['oxide_defects'].get('use_sieverts_pinhole', False)

# Pinhole kinetics: None triggers Sieverts limit inside the solver
k_diss_metal_arg = None if use_sieverts_pinhole else k_diss_metal
K_eq_metal_arg   = None if use_sieverts_pinhole else K_eq_metal

# ── Validation sweep parameters — from config ─────────────────────────────────
f_compare       = CONDITIONS['defect_fractions_compare']   # panel (A) sweep
f_defect_min    = CONDITIONS['defect_fraction_range'][0]               # panel (B) sweep
f_defect_max    = CONDITIONS['defect_fraction_range'][1]
n_defect_points = CONDITIONS['n_points']
f_limit_check   = CONDITIONS['defect_limit_check']       # panel (D) limit

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L3+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox     = {L_ox:.3e}  m")
print(f"    L_m      = {L_m:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Oxide surface kinetics  [config: oxide surface_kinetics]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox     = {D_ox:.3e}  m²/s")
print(f"    K_ox     = {K_ox:.3e}  mol/m³/Pa^0.5")

print(f"\n  Metal transport  [config: Arrhenius]")
print(f"    D_m      = {D_m:.3e}  m²/s")
print(f"    K_s_m    = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Metal surface kinetics  [config: metal surface_kinetics]")
print(f"    k_diss_metal = {k_diss_metal:.3e}  mol/m²/s/Pa")
print(f"    K_eq_metal   = {K_eq_metal:.3e}  Pa⁻¹")
print(f"    Pinhole mode: {'Sieverts limit' if use_sieverts_pinhole else 'Metal surface kinetics'}")

print(f"\n  Derived permeances")
print(f"    α_intact = {alpha_intact:.3e}  mol/m²/s/Pa^0.5")
print(f"    β        = {beta:.3e}  mol/m²/s/Pa^0.5")
print(f"    α/β      = {alpha_intact/beta:.3e}")

print(f"\n  Defect configuration  [config: OXIDE_DEFECTS]")
total_f = sum(v['area_fraction'] for v in defect_config.values())
for path, cfg in defect_config.items():
    extras = ''
    if 'thickness_factor' in cfg:
        extras = f"  γ = {cfg['thickness_factor']}"
        alpha_path = alpha_intact / cfg['thickness_factor']
        extras += f"  → α_crack = {alpha_path:.2e}"
    if 'diffusivity_factor' in cfg:
        extras = f"  δ = {cfg['diffusivity_factor']}"
        alpha_path = cfg['diffusivity_factor'] * alpha_intact
        extras += f"  → α_GB = {alpha_path:.2e}"
    print(f"    {path:<16} f = {cfg['area_fraction']*100:.3f}%{extras}")
print(f"    {'intact':<16} f = {(1-total_f)*100:.3f}%  (= 1 - total defect)")
print(f"    Total defect fraction = {total_f*100:.3f}%")

if total_f > 0.5:
    print(f"\n  ⚠  WARNING: total defect fraction = {total_f*100:.3f}% — unusually high")
if total_f == 0:
    print(f"\n  ⚠  WARNING: all defect fractions are zero — L3+L6 = L2+L6")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss        = {E_REF_SURFACE/1000:.3f} kJ/mol")
print(f"    E_D_ox+H_sol  = {E_REF_OXIDE/1000:.3f} kJ/mol")
print(f"    E_D+H_s       = {E_REF_METAL/1000:.3f} kJ/mol")

print("\n" + "=" * 60)

L3+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox     = 1.000e-06  m
    L_m      = 1.000e-03  m
    P_down   = 0.0e+00  Pa

  Oxide surface kinetics  [config: oxide surface_kinetics]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹
    k_recomb = 8.468e-05  mol/m²/s

  Oxide transport  [config: Arrhenius]
    D_ox     = 1.395e-17  m²/s
    K_ox     = 2.869e+02  mol/m³/Pa^0.5

  Metal transport  [config: Arrhenius]
    D_m      = 2.487e-10  m²/s
    K_s_m    = 3.834e-02  mol/m³/Pa^0.5

  Metal surface kinetics  [config: metal surface_kinetics]
    k_diss_metal = 4.611e-07  mol/m²/s/Pa
    K_eq_metal   = 8.212e-04  Pa⁻¹
    Pinhole mode: Metal surface kinetics

  Derived permeances
    α_intact = 4.002e-09  mol/m²/s/Pa^0.5
    β        = 9.534e-09  mol/m²/s/Pa^0.5
    α/β      = 4.197e-01

  Defect configuration  [config: OXIDE_DEFECTS]
    pinhole          f = 1.000%
    crack            f = 0.500%  γ = 0.1  → α_crack = 4.00e-08
    grain_boundary   f 

In [62]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop ──────────────────────────────────────────────────────────
rows              = []
J_arr             = []
J_intact_arr      = []
J_pinhole_arr     = []
J_crack_arr       = []
J_gb_arr          = []
frac_intact_arr   = []
frac_pinhole_arr  = []
frac_crack_arr    = []
frac_gb_arr       = []
enhancement_arr   = []
dominant_path_arr = []
frac_surface_arr  = []
frac_oxide_arr    = []
frac_metal_arr    = []
rate_lim_arr      = []

for P_up in P_RANGE_ARR:
    try:
        r = calculate_mixed_defect_flux_L6(
            P_up, P_down, L_m,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m,
            defect_config,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
        )

        J_total      = r['J_total']
        enhancement  = r['enhancement_factor']
        dom_path     = r['dominant_path']
        fb           = r['flux_breakdown']

        # Per-path flux fractions
        frac_intact  = fb.get('intact',         {}).get('fraction_of_total', 0.0)
        frac_pinhole = fb.get('pinhole',         {}).get('fraction_of_total', 0.0)
        frac_crack   = fb.get('crack',           {}).get('fraction_of_total', 0.0)
        frac_gb      = fb.get('grain_boundary',  {}).get('fraction_of_total', 0.0)

        # Per-path absolute flux
        J_intact_v   = fb.get('intact',         {}).get('flux', np.nan)
        J_pinhole_v  = fb.get('pinhole',         {}).get('flux', np.nan)
        J_crack_v    = fb.get('crack',           {}).get('flux', np.nan)
        J_gb_v       = fb.get('grain_boundary',  {}).get('flux', np.nan)

        # Resistance fractions from dominant path
        if dom_path == 'intact':
            res = r['intact_path']['resistances']
        elif dom_path in r.get('defect_paths', {}):
            res = r['defect_paths'][dom_path]['path_result']['resistances']
        else:
            # Mixed — use intact path as proxy
            res = r['intact_path']['resistances']

        fs = res.get('fraction_surface', np.nan)
        fo = res.get('fraction_oxide',   np.nan)
        fm = res.get('fraction_metal',   np.nan)
        rl = r.get('system_rate_limiting', 'mixed')

        J_arr.append(J_total)
        J_intact_arr.append(J_intact_v)
        J_pinhole_arr.append(J_pinhole_v)
        J_crack_arr.append(J_crack_v)
        J_gb_arr.append(J_gb_v)
        frac_intact_arr.append(frac_intact)
        frac_pinhole_arr.append(frac_pinhole)
        frac_crack_arr.append(frac_crack)
        frac_gb_arr.append(frac_gb)
        enhancement_arr.append(enhancement)
        dominant_path_arr.append(dom_path)
        frac_surface_arr.append(fs)
        frac_oxide_arr.append(fo)
        frac_metal_arr.append(fm)
        rate_lim_arr.append(rl)

        rows.append({
            'P_up (Pa)':            P_up,
            'J_total (mol/m²/s)':   J_total,
            'J_intact':             J_intact_v,
            'J_pinhole':            J_pinhole_v,
            'J_crack':              J_crack_v,
            'J_grain_boundary':     J_gb_v,
            'frac_intact (%)':      frac_intact  * 100,
            'frac_pinhole (%)':     frac_pinhole * 100,
            'frac_crack (%)':       frac_crack   * 100,
            'frac_GB (%)':          frac_gb      * 100,
            'enhancement_factor':   enhancement,
            'dominant_path':        dom_path,
            'fraction_surface (%)': fs * 100 if not np.isnan(fs) else np.nan,
            'fraction_oxide (%)':   fo * 100 if not np.isnan(fo) else np.nan,
            'fraction_metal (%)':   fm * 100 if not np.isnan(fm) else np.nan,
            'system_rate_limiting': rl.upper(),
        })

    except Exception as e:
        for lst in [J_arr, J_intact_arr, J_pinhole_arr, J_crack_arr,
                    J_gb_arr, frac_intact_arr, frac_pinhole_arr,
                    frac_crack_arr, frac_gb_arr, enhancement_arr,
                    frac_surface_arr, frac_oxide_arr, frac_metal_arr]:
            lst.append(np.nan)
        dominant_path_arr.append('error')
        rate_lim_arr.append('error')
        rows.append({'P_up (Pa)': P_up, 'system_rate_limiting': 'ERROR',
                     'Error': str(e)})

# ── Convert to arrays ─────────────────────────────────────────────────────────
J_arr            = np.array(J_arr)
J_intact_arr     = np.array(J_intact_arr)
J_pinhole_arr    = np.array(J_pinhole_arr)
J_crack_arr      = np.array(J_crack_arr)
J_gb_arr         = np.array(J_gb_arr)
frac_intact_arr  = np.array(frac_intact_arr)
frac_pinhole_arr = np.array(frac_pinhole_arr)
frac_crack_arr   = np.array(frac_crack_arr)
frac_gb_arr      = np.array(frac_gb_arr)
enhancement_arr  = np.array(enhancement_arr)
frac_surface_arr = np.array(frac_surface_arr)
frac_oxide_arr   = np.array(frac_oxide_arr)
frac_metal_arr   = np.array(frac_metal_arr)
rate_lim_arr     = np.array(rate_lim_arr)
dominant_path_arr= np.array(dominant_path_arr)
valid            = ~np.isnan(J_arr)

# ── Rate-limiting classification (dominant path) ──────────────────────────────
rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_oxide_arr   > 0.5, 'oxide',
         np.where(frac_metal_arr   > 0.5, 'metal', 'mixed')))

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & valid)
n_metal   = np.sum((frac_metal_arr   > 0.5) & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) &
    (frac_metal_arr   <= 0.5) & valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & valid
metal_dom_mask = (frac_metal_arr   > 0.90) & valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]
P_metal_dom    = P_RANGE_ARR[metal_dom_mask]

# ── Arrhenius pressure selection ──────────────────────────────────────────────
P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Dominant path summary ─────────────────────────────────────────────────────
dom_counts = {}
for p in ['intact', 'pinhole', 'crack', 'grain_boundary', 'mixed', 'error']:
    dom_counts[p] = np.sum(dominant_path_arr == p)

# ── Crossover pressures (diagnostic) ─────────────────────────────────────────
def find_crossover(frac_a, frac_b, valid_mask):
    vc   = valid_mask & ~np.isnan(frac_a) & ~np.isnan(frac_b)
    if np.sum(vc) < 2:
        return None
    diff = frac_a[vc] - frac_b[vc]
    Pv   = P_RANGE_ARR[vc]
    sc   = np.where(np.diff(np.sign(diff)))[0]
    if len(sc) == 0:
        return None
    idx = sc[0]
    return Pv[idx] if diff[idx] > 0 else None

P_cross_surf_oxide = find_crossover(frac_surface_arr, frac_oxide_arr, valid)
P_cross_surf_metal = find_crossover(frac_surface_arr, frac_metal_arr, valid)
P_cross_oxide_metal= find_crossover(frac_oxide_arr,   frac_metal_arr, valid)

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L3+L6 REGIME SUMMARY  (T = {T_op-273.15:.3f}°C, {np.sum(valid)} valid points)")
print("=" * 60)

print(f"\n  Resistance regime (dominant path fractions > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Oxide-limited:    {n_oxide:3d} points")
print(f"    Metal-limited:    {n_metal:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant oxide path")
for path, count in dom_counts.items():
    if count > 0:
        print(f"    {path:<18} {count:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
for label, dom, P_ea in [
    ('Surface', P_surf_dom,  P_for_surface_Ea),
    ('Oxide',   P_oxide_dom, P_for_oxide_Ea),
    ('Metal',   P_metal_dom, P_for_metal_Ea),
]:
    if len(dom) > 3:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"P = {dom.min():.1e} – {dom.max():.1e} Pa  "
              f"→ P_for_{label.lower()}_Ea = {P_ea:.2e} Pa")
    else:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"(insufficient — {label.lower()} Ea not extractable)")

print(f"\n  Crossover pressures (diagnostic only)")
for desc, P_co in [
    ('Surface → Oxide', P_cross_surf_oxide),
    ('Surface → Metal', P_cross_surf_metal),
    ('Oxide   → Metal', P_cross_oxide_metal),
]:
    val = f'{P_co:.2e} Pa' if P_co is not None else 'not found'
    print(f"    {desc}:  {val}")

print(f"\n  Enhancement factor range:")
valid_enh = enhancement_arr[valid & ~np.isnan(enhancement_arr)]
if len(valid_enh) > 0:
    print(f"    Min = {valid_enh.min():.3f}  Max = {valid_enh.max():.3f}")

print(f"\n  J range:  {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
print("=" * 60)

L3+L6 REGIME SUMMARY  (T = 599.850°C, 40 valid points)

  Resistance regime (dominant path fractions > 50%)
    Surface-limited:   20 points
    Oxide-limited:     19 points
    Metal-limited:      0 points
    Mixed:              1 points

  Dominant oxide path
    intact              26 points
    mixed               14 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  18 points  P = 1.0e-07 – 1.3e-01 Pa  → P_for_surface_Ea = 1.22e-04 Pa
    Oxide-dominant:   0 points  (insufficient — oxide Ea not extractable)
    Metal-dominant:   0 points  (insufficient — metal Ea not extractable)

  Crossover pressures (diagnostic only)
    Surface → Oxide:  1.51e+00 Pa
    Surface → Metal:  3.46e+00 Pa
    Oxide   → Metal:  not found

  Enhancement factor range:
    Min = 1.040  Max = 2.935

  J range:  6.96e-16 → 8.79e-06 mol/m²/s


In [ ]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation, one figure per panel
#   flux        flux against pressure for several total defect fractions (f = 0 to f = 1)
#   defects     flux against the fraction of each defect type alone, at CONDITIONS['P_upstream']
#   resistance  resistance fractions of the dominant path against pressure
#   limit       limit check: f → 0 recovers L2+L6
# The figures are drawn by calculations/surface_figures.py in the house style;
# this cell computes what they show.
# =============================================================================

# ── Flux against pressure: defect fraction sweep ─────────────────────────────
# Local pressure array — fixed length, never captured from outer scope
P_A = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)

# f=0 baseline — perfect oxide (L2+L6)
J_baseline = np.full(len(P_A), np.nan)
for i, P_up in enumerate(P_A):
    try:
        r = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m
        )
        J_baseline[i] = r['J_ss']
    except Exception:
        pass
v_base   = ~np.isnan(J_baseline) & (J_baseline > 0)
flux_curves = [('baseline', 0.0, J_baseline)]

# Intermediate defect fractions: the configured mix of defect types, scaled to each total
base_total = sum(v['area_fraction'] for v in defect_config.values())
for f_total in (0.001, 0.01, 0.1):
    if base_total == 0:
        continue
    scale     = f_total / base_total
    dc_scaled = {
        path: {**cfg, 'area_fraction': cfg['area_fraction'] * scale}
        for path, cfg in defect_config.items()
    }
    J_f = np.full(len(P_A), np.nan)       # pre-allocated: cannot grow beyond len(P_A)
    for i, P_up in enumerate(P_A):
        try:
            r = calculate_mixed_defect_flux_L6(
                P_up, P_down, L_m,
                k_diss, K_eq,
                D_ox, K_ox, L_ox,
                D_m, K_s_m,
                dc_scaled,
                k_diss_metal=k_diss_metal_arg,
                K_eq_metal=K_eq_metal_arg,
            )
            J_f[i] = r['J_total']
        except Exception:
            pass    # array already filled with nan at index i
    flux_curves.append(('fraction', f_total, J_f))

# f=1 — fully defective oxide: the dominant defect type (highest area fraction in config) everywhere
dom_defect_type = max(
    defect_config.keys(),
    key=lambda k: defect_config[k]['area_fraction']
)
if dom_defect_type == 'pinhole':
    alpha_f1 = np.inf
elif dom_defect_type == 'crack':
    alpha_f1 = alpha_intact / defect_config['crack'].get('thickness_factor', 0.1)
elif dom_defect_type == 'grain_boundary':
    alpha_f1 = (defect_config['grain_boundary'].get('diffusivity_factor', 100.0)
                * alpha_intact)

J_f1 = np.full(len(P_A), np.nan)
for i, P_up in enumerate(P_A):
    try:
        r = calculate_path_flux_L6(
            P_up, P_down, L_m,
            k_diss, K_eq,
            alpha_f1,
            D_m, K_s_m,
            path_type=dom_defect_type,
            k_diss_metal=k_diss_metal_arg if dom_defect_type == 'pinhole' else None,
            K_eq_metal=K_eq_metal_arg     if dom_defect_type == 'pinhole' else None,
        )
        J_f1[i] = r['flux']
    except Exception:
        pass
flux_curves.append(('full', dom_defect_type, J_f1))

# Net slope — f=0 baseline
slope_net = SF.net_slope(P_A[v_base], J_baseline[v_base])

# ── Flux against defect fraction: each defect type alone ─────────────────────
P_fixed_B  = CONDITIONS['P_upstream']
f_sweep    = np.logspace(
    np.log10(f_defect_min),
    np.log10(f_defect_max),
    n_defect_points
)

# L2+L6 reference at P_fixed_B
try:
    r_ref = solve_steady_state_flux_direct(
        P_fixed_B, P_down, L_m,
        k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m
    )
    J_L2L6_ref = r_ref['J_ss']
except Exception:
    J_L2L6_ref = np.nan

defect_types = [
    ('pinhole',        {'area_fraction': 0}),
    ('crack',          {'area_fraction': 0,
                        'thickness_factor': SIM['oxide_defects'].get('thickness_factor', 0.1)}),
    ('grain_boundary', {'area_fraction': 0,
                        'diffusivity_factor': SIM['oxide_defects'].get('diffusivity_factor', 100.0)}),
]
defect_sweeps, f_cross_by_type = [], {}
for defect_type, base_cfg in defect_types:
    J_sweep = []
    for f in f_sweep:
        dc_single = {defect_type: {**base_cfg, 'area_fraction': f}}
        try:
            r = calculate_mixed_defect_flux_L6(
                P_fixed_B, P_down, L_m,
                k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
                dc_single,
                k_diss_metal=k_diss_metal_arg if defect_type == 'pinhole' else None,
                K_eq_metal=K_eq_metal_arg   if defect_type == 'pinhole' else None,
            )
            J_sweep.append(r['J_total'])
        except Exception:
            J_sweep.append(np.nan)
    J_sweep = np.array(J_sweep)
    defect_sweeps.append((defect_type, J_sweep))

    # Crossover fraction where the flux first exceeds the L2+L6 reference
    v_sw = ~np.isnan(J_sweep) & (J_sweep > 0)
    f_cross_by_type[defect_type] = None
    if not np.isnan(J_L2L6_ref) and np.any(v_sw):
        cross = np.where(J_sweep[v_sw] > J_L2L6_ref)[0]
        if len(cross) > 0:
            f_cross_by_type[defect_type] = f_sweep[v_sw][cross[0]]

# ── Limit check: f→0 recovers L2+L6 ──────────────────────────────────────────
dc_zero = {path: {**cfg, 'area_fraction': f_limit_check} for path, cfg in defect_config.items()}

J_f_zero = []          # L3+L6 at f→0
J_L2L6   = []          # L2+L6 reference
for P_up in P_RANGE_ARR:
    try:
        r = calculate_mixed_defect_flux_L6(
            P_up, P_down, L_m,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            dc_zero,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
        )
        J_f_zero.append(r['J_total'])
    except Exception:
        J_f_zero.append(np.nan)
    try:
        r = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m
        )
        J_L2L6.append(r['J_ss'])
    except Exception:
        J_L2L6.append(np.nan)
J_f_zero = np.array(J_f_zero)
J_L2L6   = np.array(J_L2L6)

valid_parity = (~np.isnan(J_f_zero) & ~np.isnan(J_L2L6) &
                (J_f_zero > 0) & (J_L2L6 > 0))
rel_errors = (np.abs(J_f_zero[valid_parity] - J_L2L6[valid_parity])
              / J_L2L6[valid_parity] * 100)
max_err, mean_err = rel_errors.max(), rel_errors.mean()

# ── Figures ──────────────────────────────────────────────────────────────────
panels = SF.l3l6_panels(
    P_A, flux_curves, slope_net, frac_surface_arr, frac_oxide_arr, frac_metal_arr,
    [('S→O', P_cross_surf_oxide), ('O→M', P_cross_oxide_metal)],
    {'P': P_fixed_B, 'f': f_sweep, 'J_ref': J_L2L6_ref, 'types': defect_sweeps, 'f_cross': f_cross_by_type},
    J_L2L6, J_f_zero, f_limit=f_limit_check, T_C=T_op - 273.15, total_fraction=base_total,
)
save_panels(panels, "L3L6_core_validation", FIGS, module=SF.MODULE)

print("\n" + "=" * 60)
print("L3+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
print(f"  Net log-log slope:     {slope_net:.4f}")
print(f"  Enhancement range:     "
      f"{np.nanmin(enhancement_arr):.3f} → {np.nanmax(enhancement_arr):.3f}")
print(f"  Limit check error:     {max_err:.2e}%  "
      f"({'✓' if max_err < 1.0 else '⚠'})")
if not np.isnan(J_L2L6_ref):
    print(f"\n  Flux at P={P_fixed_B:.0e} Pa:")
    print(f"    L2+L6 baseline:  {J_L2L6_ref:.3e} mol/m²/s")
    for defect_type, _ in defect_types:
        cfg = defect_config.get(defect_type, {})
        f_v = cfg.get('area_fraction', 0)
        if f_v > 0:
            print(f"    {defect_type:<16} f={f_v*100:.2f}%: "
                  f"enhancement estimated from sweep")

In [65]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('system_rate_limiting', 'dominant_path', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'enhancement' in col.lower():
            df_display[col] = df_display[col].round(4)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['system_rate_limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['system_rate_limiting'] == 'OXIDE').sum()
n_metal_df   = (df['system_rate_limiting'] == 'METAL').sum()
n_mixed_df   = (df['system_rate_limiting'] == 'MIXED').sum()
n_error_df   = (df['system_rate_limiting'] == 'ERROR').sum()

n_intact_df  = (df['dominant_path'] == 'intact').sum()
n_pin_df     = (df['dominant_path'] == 'pinhole').sum()
n_crack_df   = (df['dominant_path'] == 'crack').sum()
n_gb_df      = (df['dominant_path'] == 'grain_boundary').sum()

print("=" * 60)
print("L3+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:       {OXIDE_KEY}")
print(f"  Metal:       {METAL_KEY}")
print(f"  T:           {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:        {L_ox*1e6:.3f} μm")
print(f"  L_m:         {L_m*1e3:.3f} mm")
print(f"  k_diss:      {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:        {K_eq:.3e}  Pa⁻¹")
print(f"  α_intact:    {alpha_intact:.3e}  mol/m²/s/Pa^0.5")
print(f"  β:           {beta:.3e}  mol/m²/s/Pa^0.5")

print(f"\n  Defect configuration")
for path, cfg in defect_config.items():
    print(f"    {path:<16} f = {cfg['area_fraction']*100:.3f}%")

print(f"\n  Total rows:          {len(df)}")
print(f"\n  Resistance regime (dominant path):")
print(f"    Surface-limited:   {n_surface_df}")
print(f"    Oxide-limited:     {n_oxide_df}")
print(f"    Metal-limited:     {n_metal_df}")
print(f"    Mixed:             {n_mixed_df}")
if n_error_df > 0:
    print(f"    Errors:            {n_error_df}  ⚠")

print(f"\n  Dominant oxide path:")
print(f"    Intact:            {n_intact_df}")
print(f"    Pinhole:           {n_pin_df}")
print(f"    Crack:             {n_crack_df}")
print(f"    Grain boundary:    {n_gb_df}")

print(f"\n  Enhancement factor:")
valid_enh = df['enhancement_factor'].dropna()
if len(valid_enh) > 0:
    print(f"    Min = {valid_enh.min():.4f}")
    print(f"    Max = {valid_enh.max():.4f}")
    print(f"    Mean = {valid_enh.mean():.4f}")

print("=" * 60)

display(df_display)

L3+L6 RESULTS DATAFRAME
  Oxide:       Cr2O3_sample4
  Metal:       metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:           599.850°C  (873 K)
  L_ox:        1.000 μm
  L_m:         1.000 mm
  k_diss:      2.370e-09  mol/m²/s/Pa
  K_eq:        2.799e-05  Pa⁻¹
  α_intact:    4.002e-09  mol/m²/s/Pa^0.5
  β:           9.534e-09  mol/m²/s/Pa^0.5

  Defect configuration
    pinhole          f = 1.000%
    crack            f = 0.500%
    grain_boundary   f = 0.500%

  Total rows:          40

  Resistance regime (dominant path):
    Surface-limited:   6
    Oxide-limited:     19
    Metal-limited:     0
    Mixed:             1

  Dominant oxide path:
    Intact:            26
    Pinhole:           0
    Crack:             0
    Grain boundary:    0

  Enhancement factor:
    Min = 1.0401
    Max = 2.9349
    Mean = 1.5898


,P_up (Pa),J_total (mol/m²/s),J_intact,J_pinhole,J_crack,J_grain_boundary,frac_intact (%),frac_pinhole (%),frac_crack (%),frac_GB (%),enhancement_factor,dominant_path,fraction_surface (%),fraction_oxide (%),fraction_metal (%),system_rate_limiting
0,1.0000e-07,6.9563e-16,2.3702e-16,4.6098e-14,2.3702e-16,2.3702e-16,3.3391e+01,6.6268e+01,1.7036e-01,1.7036e-01,2.9349,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
1,2.2855e-07,1.5895e-15,5.4170e-16,1.0532e-13,5.4170e-16,5.4170e-16,3.3398e+01,6.6261e+01,1.7040e-01,1.7040e-01,2.9343,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
2,5.2233e-07,3.6311e-15,1.2380e-15,2.4055e-13,1.2380e-15,1.2380e-15,3.3413e+01,6.6246e+01,1.7047e-01,1.7047e-01,2.9330,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
3,1.1938e-06,8.2902e-15,2.8295e-15,5.4890e-13,2.8295e-15,2.8295e-15,3.3448e+01,6.6211e+01,1.7065e-01,1.7065e-01,2.9300,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
4,2.7283e-06,1.8903e-14,6.4667e-15,1.2501e-12,6.4667e-15,6.4667e-15,3.3526e+01,6.6132e+01,1.7105e-01,1.7105e-01,2.9231,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
5,6.2355e-06,4.2975e-14,1.4779e-14,2.8343e-12,1.4779e-14,1.4779e-14,3.3703e+01,6.5953e+01,1.7195e-01,1.7195e-01,2.9078,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
6,1.4251e-05,9.7091e-14,3.3777e-14,6.3652e-12,3.3778e-14,3.3778e-14,3.4093e+01,6.5559e+01,1.7395e-01,1.7395e-01,2.8745,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
7,3.2570e-05,2.1662e-13,7.7196e-14,1.4020e-11,7.7198e-14,7.7198e-14,3.4924e+01,6.4720e+01,1.7819e-01,1.7819e-01,2.8061,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
8,7.4438e-05,4.7276e-13,1.7642e-13,2.9810e-11,1.7643e-13,1.7643e-13,3.6572e+01,6.3055e+01,1.8660e-01,1.8660e-01,2.6797,mixed,99.99,0.00,0.00,MIXED (MULTIPLE PATHS)
9,1.7013e-04,1.0001e-12,4.0318e-13,6.0100e-11,4.0322e-13,4.0322e-13,3.9506e+01,6.0091e+01,2.0158e-01,2.0158e-01,2.4806,mixed,99.99,0.01,0.00,MIXED (MULTIPLE PATHS)


### L2a + L4 + L6: Mixed Surface Kinetics + Mixed Bulk Diffusion

In [ ]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import (
    calculate_defective_metal_flux_L6,
    solve_steady_state_flux_direct,
)
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS, MICROSTRUCTURE, SUGGESTED_RANGES_LEVEL5,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM       = build_simulation_config()

METAL_KEY = SIM['metal_name']
OXIDE_KEY = SIM['oxide_name']
METAL     = METALS[METAL_KEY]
OXIDE     = OXIDES[OXIDE_KEY]
SK_OX     = OXIDE['surface_kinetics']

# ── Pressure sweep ────────────────────────────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']
N_T         = CONDITIONS['n_points']

# ── Arrhenius helper functions — oxide ───────────────────────────────────────
def get_k_diss_at_T(T_K):
    return SK_OX['k_diss_ref'] * np.exp(
        (-SK_OX['E_diss'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_K_eq_at_T(T_K):
    return SK_OX['K_eq_ref'] * np.exp(
        (-SK_OX['H_eq'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_D_ox_at_T(T_K):
    return OXIDE['D_ox_ref'] * np.exp(
        (-OXIDE['E_D_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

def get_K_ox_at_T(T_K):
    return OXIDE['K_ox_ref'] * np.exp(
        (-OXIDE['H_sol_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

# ── Arrhenius helper functions — metal ───────────────────────────────────────
def get_D_m_at_T(T_K):
    return METAL['D_ref'] * np.exp(
        (-METAL['E_D'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_K_s_m_at_T(T_K):
    return METAL['K_s_ref'] * np.exp(
        (-METAL['H_s'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

# ── D_eff helper — correct signature ─────────────────────────────────────────
def get_D_eff_at_T(T_K, mode, microstructure):
    """
    Compute effective metal diffusivity at T_K for a given mode.
    Uses a reference condition (P=1e5, P_down=0) and extracts D_eff.
    Falls back to D_lattice if solver fails.
    """
    try:
        r = calculate_defective_metal_flux_L6(
            1e5, 0,
            SIM['L_metal'],               # thickness
            T_K,                          # temperature — 4th positional
            get_k_diss_at_T(T_K),
            get_K_eq_at_T(T_K),
            get_D_ox_at_T(T_K),
            get_K_ox_at_T(T_K),
            SIM['L_oxide'],
            get_D_m_at_T(T_K),           # D_lattice
            get_K_s_m_at_T(T_K),
            microstructure_params=microstructure,
            mode=mode,
        )
        return r.get('D_eff', get_D_m_at_T(T_K))
    except Exception:
        return get_D_m_at_T(T_K)

# ── Known reference activation energies ──────────────────────────────────────
E_REF_SURFACE = SK_OX['E_diss']
E_REF_OXIDE   = OXIDE['E_D_ox'] + OXIDE['H_sol_ox']
E_REF_METAL   = METAL['E_D']    + METAL['H_s']

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Metal:           {METAL_KEY}")
print(f"  Oxide:           {OXIDE_KEY}")
print(f"  E_ref surface:   {E_REF_SURFACE/1000:.3f} kJ/mol  (E_diss)")
print(f"  E_ref oxide:     {E_REF_OXIDE/1000:.3f} kJ/mol  (E_D_ox + H_sol_ox)")
print(f"  E_ref metal:     {E_REF_METAL/1000:.3f} kJ/mol  (E_D + H_s)")
print(f"  P range:         {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:         {T_RANGE[0]} – {T_RANGE[1]} K")
print(f"  Valid modes:     none, gb_only, trapping_only, both")

In [67]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
L_m    = SIM['L_metal']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op ──────────────────────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Metal lattice transport at T_op ───────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Microstructure — from config ──────────────────────────────────────────────
microstructure = SIM['microstructure']
mode           = microstructure.get('mode', 'both')
grain_size     = microstructure['grain_size']
gb_thickness   = microstructure['gb_thickness']
gb_type        = microstructure.get('gb_type', 'LAGB')
lattice_density= microstructure['lattice_density']
trap_list      = microstructure['trap_list']

# ── D_eff at T_op — all four modes ───────────────────────────────────────────
D_eff_modes = {}
for m in ('none', 'gb', 'traps', 'both'):
    D_eff_modes[m] = get_D_eff_at_T(T_op, m, microstructure)

D_eff = D_eff_modes[mode]    # operating mode D_eff

# ── Derived permeances ────────────────────────────────────────────────────────
alpha    = D_ox  * K_ox / L_ox
beta_L   = D_m   * K_s_m / L_m    # lattice only
beta_eff = D_eff * K_s_m / L_m    # effective (with microstructure)

# ── Validation sweep parameters — from config ─────────────────────────────────
grain_size_sweep  = np.logspace(*np.log10(SUGGESTED_RANGES_LEVEL5['grain_size']), CONDITIONS['n_points'])
T_grain_size      = CONDITIONS['T_operating']

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L2+L4+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox     = {L_ox:.3e}  m")
print(f"    L_m      = {L_m:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Surface kinetics  [config: oxide surface_kinetics]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox     = {D_ox:.3e}  m²/s")
print(f"    K_ox     = {K_ox:.3e}  mol/m³/Pa^0.5")

print(f"\n  Metal lattice transport  [config: Arrhenius]")
print(f"    D_m      = {D_m:.3e}  m²/s")
print(f"    K_s_m    = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Microstructure  [config: MICROSTRUCTURE]")
print(f"    mode             = {mode}")
print(f"    grain_size       = {grain_size*1e6:.3f} μm")
print(f"    gb_thickness     = {gb_thickness*1e9:.3f} nm")
print(f"    gb_type          = {gb_type}")
print(f"    lattice_density  = {lattice_density:.3e}  m⁻³")
print(f"    traps:")
for trap in trap_list:
    print(f"      {trap['name']:<20} "
          f"E_b = {trap['binding_energy']/F:.2f} eV  "
          f"n = {trap['density']:.2e} m⁻³")

print(f"\n  D_eff at T_op — all modes  [config: MICROSTRUCTURE]")
for m, D in D_eff_modes.items():
    ratio = D / D_m
    marker = ' ← active' if m == mode else ''
    print(f"    mode={m:<6}  D_eff = {D:.3e} m²/s  "
          f"D_eff/D_m = {ratio:.4f}{marker}")

print(f"\n  Derived permeances")
print(f"    α        = {alpha:.3e}  mol/m²/s/Pa^0.5  (oxide)")
print(f"    β_latt   = {beta_L:.3e}  mol/m²/s/Pa^0.5  (lattice only)")
print(f"    β_eff    = {beta_eff:.3e}  mol/m²/s/Pa^0.5  (mode={mode})")
print(f"    α/β_eff  = {alpha/beta_eff:.3e}")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss        = {E_REF_SURFACE/1000:.3f} kJ/mol")
print(f"    E_D_ox+H_sol  = {E_REF_OXIDE/1000:.3f} kJ/mol")
print(f"    E_D+H_s       = {E_REF_METAL/1000:.3f} kJ/mol")

if mode == 'none':
    print(f"\n  ⚠  mode='none' — microstructure has no effect. "
          f"L2+L4+L6 = L2+L6 at this setting.")
if D_eff / D_m > 10:
    print(f"\n  ⚠  D_eff/D_m = {D_eff/D_m:.3f} — "
          f"GB short-circuit dominates at T_op.")
if D_eff / D_m < 0.1:
    print(f"\n  ⚠  D_eff/D_m = {D_eff/D_m:.3f} — "
          f"trapping strongly suppresses diffusion at T_op.")

print("\n" + "=" * 60)

L2+L4+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox     = 1.000e-06  m
    L_m      = 1.000e-03  m
    P_down   = 0.0e+00  Pa

  Surface kinetics  [config: oxide surface_kinetics]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹
    k_recomb = 8.468e-05  mol/m²/s

  Oxide transport  [config: Arrhenius]
    D_ox     = 1.395e-17  m²/s
    K_ox     = 2.869e+02  mol/m³/Pa^0.5

  Metal lattice transport  [config: Arrhenius]
    D_m      = 2.487e-10  m²/s
    K_s_m    = 3.834e-02  mol/m³/Pa^0.5

  Microstructure  [config: MICROSTRUCTURE]
    mode             = both
    grain_size       = 100.000 μm
    gb_thickness     = 0.500 nm
    gb_type          = LAGB
    lattice_density  = 8.774e+28  m⁻³
    traps:
      vacancies            E_b = 0.43 eV  n = 1.00e+26 m⁻³
      dislocations         E_b = 0.20 eV  n = 8.16e+12 m⁻³
      grain_boundaries     E_b = 0.27 eV  n = 6.00e+14 m⁻³
      Carbides             E_b = 0.27 eV  n = 2.00e+25 m⁻³

  D_eff at T_op — a

calculations/defective_metal.py:664: UserWarning: Temperature 873.0 K is outside the validated range [1023.0, 1223.0] K of metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985's diffusion data. The GB enhancement is extrapolated.
  warnings.warn(


In [68]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
L_m    = SIM['L_metal']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op ──────────────────────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Metal lattice transport at T_op ───────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Microstructure — from config ──────────────────────────────────────────────
microstructure = SIM['microstructure']
mode           = microstructure.get('mode', 'both')
grain_size     = microstructure['grain_size']
gb_thickness   = microstructure['gb_thickness']
gb_type        = microstructure.get('gb_type', 'LAGB')
lattice_density= microstructure['lattice_density']
trap_list      = microstructure['trap_list']

# ── D_eff at T_op — all four modes (correct mode strings) ────────────────────
D_eff_modes = {}
for m in ('none', 'gb_only', 'trapping_only', 'both'):
    D_eff_modes[m] = get_D_eff_at_T(T_op, m, microstructure)

D_eff = D_eff_modes[mode]    # operating mode D_eff

# ── Derived permeances ────────────────────────────────────────────────────────
alpha    = D_ox  * K_ox / L_ox
beta_L   = D_m   * K_s_m / L_m    # lattice only
beta_eff = D_eff * K_s_m / L_m    # effective (with microstructure)

# ── Validation sweep parameters — from config ─────────────────────────────────
grain_size_sweep = np.logspace(*np.log10(SUGGESTED_RANGES_LEVEL5['grain_size']), CONDITIONS['n_points'])
T_grain_size     = CONDITIONS['T_operating']

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L2+L4+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox     = {L_ox:.3e}  m")
print(f"    L_m      = {L_m:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Surface kinetics  [config: oxide surface_kinetics]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox     = {D_ox:.3e}  m²/s")
print(f"    K_ox     = {K_ox:.3e}  mol/m³/Pa^0.5")

print(f"\n  Metal lattice transport  [config: Arrhenius]")
print(f"    D_m      = {D_m:.3e}  m²/s")
print(f"    K_s_m    = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Microstructure  [config: MICROSTRUCTURE]")
print(f"    mode             = {mode}")
print(f"    grain_size       = {grain_size*1e6:.3f} μm")
print(f"    gb_thickness     = {gb_thickness*1e9:.3f} nm")
print(f"    gb_type          = {gb_type}")
print(f"    lattice_density  = {lattice_density:.3e}  m⁻³")
print(f"    traps:")
for trap in trap_list:
    print(f"      {trap['name']:<20} "
          f"E_b = {trap['binding_energy']/F:.2f} eV  "
          f"n = {trap['density']:.2e} m⁻³")

print(f"\n  D_eff at T_op — all modes")
for m, D in D_eff_modes.items():
    ratio  = D / D_m
    marker = ' ← active' if m == mode else ''
    print(f"    mode={m:<14}  D_eff = {D:.3e} m²/s  "
          f"D_eff/D_m = {ratio:.4f}{marker}")

print(f"\n  Derived permeances")
print(f"    α        = {alpha:.3e}  mol/m²/s/Pa^0.5  (oxide)")
print(f"    β_latt   = {beta_L:.3e}  mol/m²/s/Pa^0.5  (lattice only)")
print(f"    β_eff    = {beta_eff:.3e}  mol/m²/s/Pa^0.5  (mode={mode})")
print(f"    α/β_eff  = {alpha/beta_eff:.3e}")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss        = {E_REF_SURFACE/1000:.3f} kJ/mol")
print(f"    E_D_ox+H_sol  = {E_REF_OXIDE/1000:.3f} kJ/mol")
print(f"    E_D+H_s       = {E_REF_METAL/1000:.3f} kJ/mol")

if mode == 'none':
    print(f"\n  ⚠  mode='none' — microstructure disabled. L2+L4+L6 = L2+L6.")
if D_eff / D_m > 10:
    print(f"\n  ⚠  D_eff/D_m = {D_eff/D_m:.3f} — GB short-circuit dominates at T_op.")
if D_eff / D_m < 0.1:
    print(f"\n  ⚠  D_eff/D_m = {D_eff/D_m:.3f} — trapping strongly suppresses diffusion.")

print("\n" + "=" * 60)

L2+L4+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox     = 1.000e-06  m
    L_m      = 1.000e-03  m
    P_down   = 0.0e+00  Pa

  Surface kinetics  [config: oxide surface_kinetics]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹
    k_recomb = 8.468e-05  mol/m²/s

  Oxide transport  [config: Arrhenius]
    D_ox     = 1.395e-17  m²/s
    K_ox     = 2.869e+02  mol/m³/Pa^0.5

  Metal lattice transport  [config: Arrhenius]
    D_m      = 2.487e-10  m²/s
    K_s_m    = 3.834e-02  mol/m³/Pa^0.5

  Microstructure  [config: MICROSTRUCTURE]
    mode             = both
    grain_size       = 100.000 μm
    gb_thickness     = 0.500 nm
    gb_type          = LAGB
    lattice_density  = 8.774e+28  m⁻³
    traps:
      vacancies            E_b = 0.43 eV  n = 1.00e+26 m⁻³
      dislocations         E_b = 0.20 eV  n = 8.16e+12 m⁻³
      grain_boundaries     E_b = 0.27 eV  n = 6.00e+14 m⁻³
      Carbides             E_b = 0.27 eV  n = 2.00e+25 m⁻³

  D_eff at T_op — a

In [69]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop — pre-allocated fixed-length arrays ─────────────────────
rows             = []
J_arr            = np.full(len(P_RANGE_ARR), np.nan)
theta_arr        = np.full(len(P_RANGE_ARR), np.nan)
P_int_arr        = np.full(len(P_RANGE_ARR), np.nan)
D_eff_arr        = np.full(len(P_RANGE_ARR), np.nan)
frac_surface_arr = np.full(len(P_RANGE_ARR), np.nan)
frac_oxide_arr   = np.full(len(P_RANGE_ARR), np.nan)
frac_metal_arr   = np.full(len(P_RANGE_ARR), np.nan)
R_surface_arr    = np.full(len(P_RANGE_ARR), np.nan)
R_oxide_arr      = np.full(len(P_RANGE_ARR), np.nan)
R_metal_arr      = np.full(len(P_RANGE_ARR), np.nan)
R_total_arr      = np.full(len(P_RANGE_ARR), np.nan)
rate_lim_arr     = np.empty(len(P_RANGE_ARR), dtype=object)

for i, P_up in enumerate(P_RANGE_ARR):
    try:
        r = calculate_defective_metal_flux_L6(
            P_up, P_down,
            L_m,                              # thickness
            T_op,                             # temperature — 4th positional
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m,                              # D_lattice
            K_s_m,
            microstructure_params=microstructure,
            mode=mode,
        )

        J_arr[i]            = r['flux']
        theta_arr[i]        = r['theta_surface']
        P_int_arr[i]        = r['P_int']
        D_eff_arr[i]        = r.get('D_eff', D_m)
        frac_surface_arr[i] = r['resistances']['fraction_surface']
        frac_oxide_arr[i]   = r['resistances']['fraction_oxide']
        frac_metal_arr[i]   = r['resistances']['fraction_metal']
        R_surface_arr[i]    = r['resistances']['R_surface']
        R_oxide_arr[i]      = r['resistances']['R_oxide']
        R_metal_arr[i]      = r['resistances']['R_metal']
        R_total_arr[i]      = r['resistances']['R_total']
        rate_lim_arr[i]     = r['rate_limiting']

        rows.append({
            'P_up (Pa)':            P_up,
            'P_int (Pa)':           r['P_int'],
            'J_ss (mol/m²/s)':      r['flux'],
            'θ_surface':            r['theta_surface'],
            'D_eff (m²/s)':         r.get('D_eff', D_m),
            'D_eff/D_m':            r.get('D_eff', D_m) / D_m,
            'α_oxide':              r.get('alpha', alpha),
            'β_eff':                r.get('beta_eff', beta_eff),
            'R_surface':            r['resistances']['R_surface'],
            'R_oxide':              r['resistances']['R_oxide'],
            'R_metal':              r['resistances']['R_metal'],
            'R_total':              r['resistances']['R_total'],
            'fraction_surface (%)': r['resistances']['fraction_surface'] * 100,
            'fraction_oxide (%)':   r['resistances']['fraction_oxide']   * 100,
            'fraction_metal (%)':   r['resistances']['fraction_metal']   * 100,
            'Rate-Limiting':        r['rate_limiting'].upper(),
            'mode':                 mode,
        })

    except Exception as e:
        rate_lim_arr[i] = 'error'
        rows.append({
            'P_up (Pa)':     P_up,
            'Rate-Limiting': 'ERROR',
            'mode':          mode,
            'Error':         str(e),
        })

# ── Derived arrays ────────────────────────────────────────────────────────────
valid  = ~np.isnan(J_arr)

rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_oxide_arr   > 0.5, 'oxide',
         np.where(frac_metal_arr   > 0.5, 'metal', 'mixed')))

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & valid)
n_metal   = np.sum((frac_metal_arr   > 0.5) & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) &
    (frac_metal_arr   <= 0.5) & valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & valid
metal_dom_mask = (frac_metal_arr   > 0.90) & valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]
P_metal_dom    = P_RANGE_ARR[metal_dom_mask]

P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Crossover pressures ───────────────────────────────────────────────────────
def find_crossover(frac_a, frac_b, valid_mask):
    vc   = valid_mask & ~np.isnan(frac_a) & ~np.isnan(frac_b)
    if np.sum(vc) < 2:
        return None
    diff = frac_a[vc] - frac_b[vc]
    Pv   = P_RANGE_ARR[vc]
    sc   = np.where(np.diff(np.sign(diff)))[0]
    if len(sc) == 0:
        return None
    idx = sc[0]
    return Pv[idx] if diff[idx] > 0 else None

P_cross_surf_oxide  = find_crossover(frac_surface_arr, frac_oxide_arr,  valid)
P_cross_surf_metal  = find_crossover(frac_surface_arr, frac_metal_arr,  valid)
P_cross_oxide_metal = find_crossover(frac_oxide_arr,   frac_metal_arr,  valid)

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L2+L4+L6 REGIME SUMMARY")
print(f"T = {T_op-273.15:.3f}°C  |  mode = {mode}  |  "
      f"{np.sum(valid)} valid points")
print("=" * 60)

print(f"\n  Resistance regime (fraction > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Oxide-limited:    {n_oxide:3d} points")
print(f"    Metal-limited:    {n_metal:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
for label, dom, P_ea in [
    ('Surface', P_surf_dom,  P_for_surface_Ea),
    ('Oxide',   P_oxide_dom, P_for_oxide_Ea),
    ('Metal',   P_metal_dom, P_for_metal_Ea),
]:
    if len(dom) > 3:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"P = {dom.min():.1e} – {dom.max():.1e} Pa  "
              f"→ P_for_{label.lower()}_Ea = {P_ea:.2e} Pa")
    else:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"(insufficient — {label.lower()} Ea not extractable)")

print(f"\n  Crossover pressures (diagnostic only)")
for desc, P_co in [
    ('Surface → Oxide', P_cross_surf_oxide),
    ('Surface → Metal', P_cross_surf_metal),
    ('Oxide   → Metal', P_cross_oxide_metal),
]:
    val = f'{P_co:.2e} Pa' if P_co is not None else 'not found'
    print(f"    {desc}:  {val}")

print(f"\n  D_eff at T_op")
print(f"    D_eff (mode={mode})  = {D_eff:.3e} m²/s")
print(f"    D_m   (lattice)     = {D_m:.3e} m²/s")
print(f"    D_eff/D_m           = {D_eff/D_m:.4f}")
d_eff_range = D_eff_arr[valid]
if len(d_eff_range) > 0:
    print(f"    D_eff range over P  = "
          f"{np.nanmin(d_eff_range):.3e} – {np.nanmax(d_eff_range):.3e} m²/s")

if np.sum(valid) > 0:
    print(f"\n  θ range:  {np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
    print(f"  J range:  {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
else:
    print(f"\n  ⚠  No valid points — check solver signature and config.")
print("=" * 60)

L2+L4+L6 REGIME SUMMARY
T = 599.850°C  |  mode = both  |  40 valid points

  Resistance regime (fraction > 50%)
    Surface-limited:   20 points
    Oxide-limited:     18 points
    Metal-limited:      0 points
    Mixed:              2 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  17 points  P = 1.0e-07 – 5.5e-02 Pa  → P_for_surface_Ea = 7.44e-05 Pa
    Oxide-dominant:   0 points  (insufficient — oxide Ea not extractable)
    Metal-dominant:   0 points  (insufficient — metal Ea not extractable)

  Crossover pressures (diagnostic only)
    Surface → Oxide:  1.51e+00 Pa
    Surface → Metal:  1.51e+00 Pa
    Oxide   → Metal:  not found

  D_eff at T_op
    D_eff (mode=both)  = 1.923e-10 m²/s
    D_m   (lattice)     = 2.487e-10 m²/s
    D_eff/D_m           = 0.7732
    D_eff range over P  = 1.923e-10 – 1.923e-10 m²/s

  θ range:  0.0000 → 0.9409
  J range:  2.37e-16 → 7.81e-06 mol/m²/s


In [ ]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation, one figure per panel
#   flux        flux against pressure for each microstructure mode of the metal
#   diffusivity D_eff/D_m against temperature for each mode
#   resistance  resistance fractions against pressure at the operating mode
#   limit       limit check: mode='none' recovers L2+L6
# The figures are drawn by calculations/surface_figures.py in the house style;
# this cell computes what they show.
# =============================================================================

# ── Flux against pressure: mode comparison ───────────────────────────────────
P_A = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
mode_curves = []
for m in ('none', 'gb_only', 'trapping_only', 'both'):
    J_m  = np.full(len(P_A), np.nan)
    fs_m = np.full(len(P_A), np.nan)
    fo_m = np.full(len(P_A), np.nan)
    fm_m = np.full(len(P_A), np.nan)
    for i, P_up in enumerate(P_A):
        try:
            r = calculate_defective_metal_flux_L6(
                P_up, P_down,
                L_m,
                T_op,
                k_diss, K_eq,
                D_ox, K_ox, L_ox,
                D_m,
                K_s_m,
                microstructure_params=microstructure,
                mode=m,
            )
            J_m[i]  = r['flux']
            fs_m[i] = r['resistances']['fraction_surface']
            fo_m[i] = r['resistances']['fraction_oxide']
            fm_m[i] = r['resistances']['fraction_metal']
        except Exception:
            pass
    mode_curves.append((m, J_m, fs_m, fo_m, fm_m))

# Net slope at the operating mode
v_net = ~np.isnan(J_arr) & (J_arr > 0)
slope_net = SF.net_slope(P_RANGE_ARR[v_net], J_arr[v_net]) if np.sum(v_net) > 2 else np.nan

# ── D_eff/D_m against temperature: mode comparison ───────────────────────────
temperatures_K = np.linspace(T_RANGE[0], T_RANGE[1], N_T)
ratio_by_mode, T_cross_by_mode = {}, {}
for m in ('none', 'gb_only', 'trapping_only', 'both'):
    ratio_arr = np.full(N_T, np.nan)
    for j, T_K in enumerate(temperatures_K):
        D_latt  = get_D_m_at_T(T_K)
        D_eff_j = get_D_eff_at_T(T_K, m, microstructure)
        if D_latt > 0:
            ratio_arr[j] = D_eff_j / D_latt
    ratio_by_mode[m] = ratio_arr
    # where the ratio crosses 1 (grain boundaries speeding diffusion past the lattice)
    T_cross_by_mode[m] = None
    v_r = ~np.isnan(ratio_arr)
    if m in ('gb_only', 'both') and np.any(v_r):
        cross = np.where(np.diff(np.sign(ratio_arr[v_r] - 1.0)))[0]
        if len(cross) > 0:
            T_cross_by_mode[m] = temperatures_K[v_r][cross[0]] - 273.15

# ── Limit check: mode='none' recovers L2+L6 ──────────────────────────────────
P_D = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
J_none = np.full(len(P_D), np.nan)
J_L2L6 = np.full(len(P_D), np.nan)
for i, P_up in enumerate(P_D):
    try:
        r_none = calculate_defective_metal_flux_L6(
            P_up, P_down,
            L_m,
            T_op,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m,
            K_s_m,
            microstructure_params=microstructure,
            mode='none',
        )
        J_none[i] = r_none['flux']
    except Exception:
        pass
    try:
        r_L2L6 = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m
        )
        J_L2L6[i] = r_L2L6['J_ss']
    except Exception:
        pass

valid_parity = (~np.isnan(J_none) & ~np.isnan(J_L2L6) &
                (J_none > 0) & (J_L2L6 > 0))
if np.any(valid_parity):
    rel_errors = (np.abs(J_none[valid_parity] - J_L2L6[valid_parity])
                  / J_L2L6[valid_parity] * 100)
    max_err, mean_err = rel_errors.max(), rel_errors.mean()
else:
    max_err = mean_err = np.nan

# ── Figures ──────────────────────────────────────────────────────────────────
panels = SF.l2l4l6_panels(
    P_A, mode_curves, mode, slope_net, frac_surface_arr, frac_oxide_arr, frac_metal_arr,
    [('S→O', P_cross_surf_oxide), ('O→M', P_cross_oxide_metal)],
    {'T_C': temperatures_K - 273.15, 'ratio': ratio_by_mode, 'T_cross': T_cross_by_mode, 'T_op_C': T_op - 273.15},
    D_eff / D_m, J_L2L6, J_none, T_C=T_op - 273.15, grain_um=grain_size * 1e6,
)
save_panels(panels, "L2L4L6_perfect_oxide_defective_metal_surface_kinetics", FIGS, module=SF.MODULE)

print("\n" + "=" * 60)
print("L2+L4+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
if not np.isnan(slope_net):
    print(f"  Net log-log slope (mode={mode}): {slope_net:.4f}")
print(f"  D_eff/D_m at T_op:")
for m, D in D_eff_modes.items():
    print(f"    mode={m:<14}  {D/D_m:.4f}")
print(f"  Limit check error (mode=none): {max_err:.2e}%  "
      f"({'✓' if not np.isnan(max_err) and max_err < 1.0 else '⚠'})")
if P_cross_surf_oxide is not None:
    print(f"  S→O crossover:  {P_cross_surf_oxide:.2e} Pa")
if P_cross_oxide_metal is not None:
    print(f"  O→M crossover:  {P_cross_oxide_metal:.2e} Pa")

In [72]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('Rate-Limiting', 'mode', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'theta' in col.lower():
            df_display[col] = df_display[col].round(6)
        elif 'D_eff/D_m' in col:
            df_display[col] = df_display[col].round(6)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['Rate-Limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['Rate-Limiting'] == 'OXIDE').sum()
n_metal_df   = (df['Rate-Limiting'] == 'METAL').sum()
n_mixed_df   = (df['Rate-Limiting'] == 'MIXED').sum()
n_error_df   = (df['Rate-Limiting'] == 'ERROR').sum()

print("=" * 60)
print("L2+L4+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:           {OXIDE_KEY}")
print(f"  Metal:           {METAL_KEY}")
print(f"  T:               {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:            {L_ox*1e6:.3f} μm")
print(f"  L_m:             {L_m*1e3:.3f} mm")
print(f"  mode:            {mode}")
print(f"  grain_size:      {grain_size*1e6:.3f} μm")
print(f"  k_diss:          {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:            {K_eq:.3e}  Pa⁻¹")
print(f"  α:               {alpha:.3e}  mol/m²/s/Pa^0.5  (oxide)")
print(f"  β_latt:          {beta_L:.3e}  mol/m²/s/Pa^0.5  (lattice)")
print(f"  β_eff:           {beta_eff:.3e}  mol/m²/s/Pa^0.5  (mode={mode})")
print(f"  D_eff/D_m:       {D_eff/D_m:.6f}")

print(f"\n  Total rows:        {len(df)}")
print(f"\n  Resistance regime:")
print(f"    Surface-limited: {n_surface_df}")
print(f"    Oxide-limited:   {n_oxide_df}")
print(f"    Metal-limited:   {n_metal_df}")
print(f"    Mixed:           {n_mixed_df}")
if n_error_df > 0:
    print(f"    Errors:          {n_error_df}  ⚠")

# D_eff statistics across pressure sweep
d_eff_col = df.get('D_eff (m²/s)', pd.Series(dtype=float))
if 'D_eff (m²/s)' in df.columns:
    d_vals = df['D_eff (m²/s)'].dropna()
    if len(d_vals) > 0:
        print(f"\n  D_eff across pressure sweep:")
        print(f"    Min:   {d_vals.min():.4e} m²/s")
        print(f"    Max:   {d_vals.max():.4e} m²/s")
        print(f"    Mean:  {d_vals.mean():.4e} m²/s")
        print(f"    D_m:   {D_m:.4e} m²/s")

if 'D_eff/D_m' in df.columns:
    ratio_vals = df['D_eff/D_m'].dropna()
    if len(ratio_vals) > 0:
        print(f"\n  D_eff/D_m range:  "
              f"{ratio_vals.min():.6f} → {ratio_vals.max():.6f}")

print("=" * 60)

display(df_display)

L2+L4+L6 RESULTS DATAFRAME
  Oxide:           Cr2O3_sample4
  Metal:           metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:               599.850°C  (873 K)
  L_ox:            1.000 μm
  L_m:             1.000 mm
  mode:            both
  grain_size:      100.000 μm
  k_diss:          2.370e-09  mol/m²/s/Pa
  K_eq:            2.799e-05  Pa⁻¹
  α:               4.002e-09  mol/m²/s/Pa^0.5  (oxide)
  β_latt:          9.534e-09  mol/m²/s/Pa^0.5  (lattice)
  β_eff:           7.372e-09  mol/m²/s/Pa^0.5  (mode=both)
  D_eff/D_m:       0.773222

  Total rows:        40

  Resistance regime:
    Surface-limited: 20
    Oxide-limited:   18
    Metal-limited:   0
    Mixed:           2

  D_eff across pressure sweep:
    Min:   1.9227e-10 m²/s
    Max:   1.9227e-10 m²/s
    Mean:  1.9227e-10 m²/s
    D_m:   2.4866e-10 m²/s

  D_eff/D_m range:  0.773222 → 0.773222


,P_up (Pa),P_int (Pa),J_ss (mol/m²/s),θ_surface,D_eff (m²/s),D_eff/D_m,α_oxide,β_eff,R_surface,R_oxide,R_metal,R_total,fraction_surface (%),fraction_oxide (%),fraction_metal (%),Rate-Limiting,mode
0,1.0000e-07,1.0337e-15,2.3702e-16,4.8345e-10,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,4.2191e+15,2.4990e+08,1.3565e+08,4.2191e+15,100.00,0.00,0.00,SURFACE,both
1,2.2855e-07,5.3992e-15,5.4170e-16,1.1049e-09,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,1.8460e+15,2.4990e+08,1.3565e+08,1.8460e+15,100.00,0.00,0.00,SURFACE,both
2,5.2233e-07,2.8202e-14,1.2380e-15,2.5252e-09,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,8.0773e+14,2.4990e+08,1.3565e+08,8.0773e+14,100.00,0.00,0.00,SURFACE,both
3,1.1938e-06,1.4731e-13,2.8295e-15,5.7713e-09,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,3.5342e+14,2.4990e+08,1.3565e+08,3.5342e+14,100.00,0.00,0.00,SURFACE,both
4,2.7283e-06,7.6944e-13,6.4667e-15,1.3190e-08,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,1.5464e+14,2.4990e+08,1.3565e+08,1.5464e+14,100.00,0.00,0.00,SURFACE,both
5,6.2355e-06,4.0190e-12,1.4779e-14,3.0145e-08,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,6.7662e+13,2.4990e+08,1.3565e+08,6.7662e+13,100.00,0.00,0.00,SURFACE,both
6,1.4251e-05,2.0992e-11,3.3777e-14,6.8895e-08,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,2.9605e+13,2.4990e+08,1.3565e+08,2.9606e+13,100.00,0.00,0.00,SURFACE,both
7,3.2570e-05,1.0965e-10,7.7195e-14,1.5746e-07,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,1.2954e+13,2.4990e+08,1.3565e+08,1.2954e+13,100.00,0.00,0.00,SURFACE,both
8,7.4438e-05,5.7268e-10,1.7642e-13,3.5985e-07,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,5.6679e+12,2.4990e+08,1.3565e+08,5.6683e+12,99.99,0.00,0.00,SURFACE,both
9,1.7013e-04,2.9908e-09,4.0317e-13,8.2235e-07,1.9227e-10,0.773222,4.0017e-09,7.3722e-09,2.4800e+12,2.4990e+08,1.3565e+08,2.4804e+12,99.98,0.01,0.01,SURFACE,both


## L3+L4+L6: Defective Oxide + Defective Metal + Surface Kinetics

In [ ]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import (
    calculate_full_model_flux_L346_v2,
    calculate_mixed_defect_flux_L6,
    calculate_defective_metal_flux_L6,
    solve_steady_state_flux_direct,
)
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS, MICROSTRUCTURE, SUGGESTED_RANGES_LEVEL5,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM       = build_simulation_config()

METAL_KEY = SIM['metal_name']
OXIDE_KEY = SIM['oxide_name']
METAL     = METALS[METAL_KEY]
OXIDE     = OXIDES[OXIDE_KEY]
SK_OX     = OXIDE['surface_kinetics']
SK_M      = METAL['surface_kinetics']

# ── Pressure sweep ────────────────────────────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']
N_T         = CONDITIONS['n_points']

# ── Arrhenius helpers — oxide surface ────────────────────────────────────────
def get_k_diss_at_T(T_K):
    return SK_OX['k_diss_ref'] * np.exp(
        (-SK_OX['E_diss'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_K_eq_at_T(T_K):
    return SK_OX['K_eq_ref'] * np.exp(
        (-SK_OX['H_eq'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_D_ox_at_T(T_K):
    return OXIDE['D_ox_ref'] * np.exp(
        (-OXIDE['E_D_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

def get_K_ox_at_T(T_K):
    return OXIDE['K_ox_ref'] * np.exp(
        (-OXIDE['H_sol_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

# ── Arrhenius helpers — metal ────────────────────────────────────────────────
def get_D_m_at_T(T_K):
    return METAL['D_ref'] * np.exp(
        (-METAL['E_D'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_K_s_m_at_T(T_K):
    return METAL['K_s_ref'] * np.exp(
        (-METAL['H_s'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_k_diss_metal_at_T(T_K):
    return SK_M['k_diss_metal_ref'] * np.exp(
        (-SK_M['E_diss_metal'] / R) * (1/T_K - 1/SK_M['T_ref'])
    )

def get_K_eq_metal_at_T(T_K):
    return SK_M['K_eq_metal_ref'] * np.exp(
        (-SK_M['H_eq_metal'] / R) * (1/T_K - 1/SK_M['T_ref'])
    )

# ── D_eff helper ──────────────────────────────────────────────────────────────
def get_D_eff_at_T(T_K, mode, microstructure):
    try:
        r = calculate_defective_metal_flux_L6(
            1e5, 0, SIM['L_metal'], T_K,
            get_k_diss_at_T(T_K), get_K_eq_at_T(T_K),
            get_D_ox_at_T(T_K),   get_K_ox_at_T(T_K), SIM['L_oxide'],
            get_D_m_at_T(T_K),    get_K_s_m_at_T(T_K),
            microstructure_params=microstructure,
            mode=mode,
        )
        return r.get('D_eff', get_D_m_at_T(T_K))
    except Exception:
        return get_D_m_at_T(T_K)

# ── Reference activation energies ────────────────────────────────────────────
E_REF_SURFACE = SK_OX['E_diss']
E_REF_OXIDE   = OXIDE['E_D_ox'] + OXIDE['H_sol_ox']
E_REF_METAL   = METAL['E_D']    + METAL['H_s']

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Metal:         {METAL_KEY}")
print(f"  Oxide:         {OXIDE_KEY}")
print(f"  E_ref surface: {E_REF_SURFACE/1000:.3f} kJ/mol")
print(f"  E_ref oxide:   {E_REF_OXIDE/1000:.3f} kJ/mol")
print(f"  E_ref metal:   {E_REF_METAL/1000:.3f} kJ/mol")
print(f"  P range:       {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:       {T_RANGE[0]} – {T_RANGE[1]} K")

In [74]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
L_m    = SIM['L_metal']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op ──────────────────────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Metal lattice transport at T_op ───────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Metal surface kinetics — for pinhole paths ────────────────────────────────
k_diss_metal = get_k_diss_metal_at_T(T_op)
K_eq_metal   = get_K_eq_metal_at_T(T_op)

# ── Microstructure from config ────────────────────────────────────────────────
microstructure = SIM['microstructure']
mode           = microstructure.get('mode', 'both')
grain_size     = microstructure['grain_size']

# ── D_eff at T_op — all four modes ───────────────────────────────────────────
D_eff_modes = {}
for m in ('none', 'gb_only', 'trapping_only', 'both'):
    D_eff_modes[m] = get_D_eff_at_T(T_op, m, microstructure)
D_eff = D_eff_modes[mode]

# ── Defect configuration from SIM ────────────────────────────────────────────
defect_config        = SIM['defect_config']
use_sieverts_pinhole = SIM['oxide_defects'].get('use_sieverts_pinhole', False)
k_diss_metal_arg     = None if use_sieverts_pinhole else k_diss_metal
K_eq_metal_arg       = None if use_sieverts_pinhole else K_eq_metal

# ── Derived ───────────────────────────────────────────────────────────────────
alpha_intact = D_ox * K_ox / L_ox
beta_L       = D_m   * K_s_m / L_m
beta_eff     = D_eff * K_s_m / L_m

# ── Validation parameters ─────────────────────────────────────────────────────
f_limit_check = CONDITIONS['defect_limit_check']

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L3+L4+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox   = {L_ox:.3e}  m")
print(f"    L_m    = {L_m:.3e}  m")
print(f"    P_down = {P_down:.1e}  Pa")

print(f"\n  Oxide surface kinetics  [config: oxide surface_kinetics]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox   = {D_ox:.3e}  m²/s")
print(f"    K_ox   = {K_ox:.3e}  mol/m³/Pa^0.5")
print(f"    α      = {alpha_intact:.3e}  mol/m²/s/Pa^0.5")

print(f"\n  Defect configuration  [config: OXIDE_DEFECTS]")
total_f = sum(v['area_fraction'] for v in defect_config.values())
for path, cfg in defect_config.items():
    extras = ''
    if 'thickness_factor' in cfg:
        extras = f"  γ={cfg['thickness_factor']}"
    if 'diffusivity_factor' in cfg:
        extras = f"  δ={cfg['diffusivity_factor']}"
    print(f"    {path:<16} f = {cfg['area_fraction']*100:.3f}%{extras}")
print(f"    {'intact':<16} f = {(1-total_f)*100:.3f}%")
print(f"    Total defect fraction = {total_f*100:.3f}%")

print(f"\n  Metal lattice transport  [config: Arrhenius]")
print(f"    D_m    = {D_m:.3e}  m²/s")
print(f"    K_s_m  = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Microstructure  [config: MICROSTRUCTURE]")
print(f"    mode       = {mode}")
print(f"    grain_size = {grain_size*1e6:.3f} μm")

print(f"\n  D_eff at T_op — all modes")
for m, D in D_eff_modes.items():
    marker = ' ← active' if m == mode else ''
    print(f"    mode={m:<14}  D_eff/D_m = {D/D_m:.4f}{marker}")

print(f"\n  Derived permeances")
print(f"    α_intact = {alpha_intact:.3e}  mol/m²/s/Pa^0.5")
print(f"    β_latt   = {beta_L:.3e}  mol/m²/s/Pa^0.5")
print(f"    β_eff    = {beta_eff:.3e}  mol/m²/s/Pa^0.5  (mode={mode})")

print(f"\n  Pinhole mode: "
      f"{'Sieverts limit' if use_sieverts_pinhole else 'Metal surface kinetics'}")

if total_f > 0.5:
    print(f"\n  ⚠  WARNING: total defect fraction = {total_f*100:.3f}%")
if total_f == 0:
    print(f"\n  ⚠  WARNING: all defect fractions zero — L3+L4+L6 = L2+L4+L6")
if mode == 'none':
    print(f"\n  ⚠  mode='none' — L3+L4+L6 = L3+L6")

print("\n" + "=" * 60)

L3+L4+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox   = 1.000e-06  m
    L_m    = 1.000e-03  m
    P_down = 0.0e+00  Pa

  Oxide surface kinetics  [config: oxide surface_kinetics]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹

  Oxide transport  [config: Arrhenius]
    D_ox   = 1.395e-17  m²/s
    K_ox   = 2.869e+02  mol/m³/Pa^0.5
    α      = 4.002e-09  mol/m²/s/Pa^0.5

  Defect configuration  [config: OXIDE_DEFECTS]
    pinhole          f = 1.000%
    crack            f = 0.500%  γ=0.1
    grain_boundary   f = 0.500%  δ=10
    intact           f = 98.000%
    Total defect fraction = 2.000%

  Metal lattice transport  [config: Arrhenius]
    D_m    = 2.487e-10  m²/s
    K_s_m  = 3.834e-02  mol/m³/Pa^0.5

  Microstructure  [config: MICROSTRUCTURE]
    mode       = both
    grain_size = 100.000 μm

  D_eff at T_op — all modes
    mode=none            D_eff/D_m = 1.0000
    mode=gb_only         D_eff/D_m = 1.0001
    mode=trapping_only   D_eff/D_m = 

calculations/defective_metal.py:664: UserWarning: Temperature 873.0 K is outside the validated range [1023.0, 1223.0] K of metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985's diffusion data. The GB enhancement is extrapolated.
  warnings.warn(


In [75]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# =============================================================================

rows             = []
J_arr            = np.full(len(P_RANGE_ARR), np.nan)
J_intact_arr     = np.full(len(P_RANGE_ARR), np.nan)
J_pinhole_arr    = np.full(len(P_RANGE_ARR), np.nan)
J_crack_arr      = np.full(len(P_RANGE_ARR), np.nan)
J_gb_arr         = np.full(len(P_RANGE_ARR), np.nan)
frac_intact_arr  = np.full(len(P_RANGE_ARR), np.nan)
frac_pinhole_arr = np.full(len(P_RANGE_ARR), np.nan)
frac_crack_arr   = np.full(len(P_RANGE_ARR), np.nan)
frac_gb_arr      = np.full(len(P_RANGE_ARR), np.nan)
enhancement_arr  = np.full(len(P_RANGE_ARR), np.nan)
D_eff_avg_arr    = np.full(len(P_RANGE_ARR), np.nan)
frac_surface_arr = np.full(len(P_RANGE_ARR), np.nan)
frac_oxide_arr   = np.full(len(P_RANGE_ARR), np.nan)
frac_metal_arr   = np.full(len(P_RANGE_ARR), np.nan)
dominant_path_arr= np.empty(len(P_RANGE_ARR), dtype=object)
rate_lim_arr     = np.empty(len(P_RANGE_ARR), dtype=object)

for i, P_up in enumerate(P_RANGE_ARR):
    try:
        r = calculate_full_model_flux_L346_v2(
            P_up, P_down, L_m, T_op,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m,
            microstructure,
            defect_config,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
            mode=mode,
        )

        fb  = r['flux_breakdown']
        dom = r['dominant_path']
        res = r.get('flux_weighted_resistances')

        J_arr[i]            = r['J_total']
        J_intact_arr[i]     = fb.get('intact',         {}).get('flux', np.nan)
        J_pinhole_arr[i]    = fb.get('pinhole',         {}).get('flux', np.nan)
        J_crack_arr[i]      = fb.get('crack',           {}).get('flux', np.nan)
        J_gb_arr[i]         = fb.get('grain_boundary',  {}).get('flux', np.nan)
        frac_intact_arr[i]  = fb.get('intact',         {}).get('fraction_of_total', 0.0)
        frac_pinhole_arr[i] = fb.get('pinhole',         {}).get('fraction_of_total', 0.0)
        frac_crack_arr[i]   = fb.get('crack',           {}).get('fraction_of_total', 0.0)
        frac_gb_arr[i]      = fb.get('grain_boundary',  {}).get('fraction_of_total', 0.0)
        enhancement_arr[i]  = r['enhancement_vs_intact']
        D_eff_avg_arr[i]    = r['D_eff_avg']
        dominant_path_arr[i]= dom
        rate_lim_arr[i]     = r['system_rate_limiting']

        if res is not None:
            frac_surface_arr[i] = res['fraction_surface']
            frac_oxide_arr[i]   = res['fraction_oxide']
            frac_metal_arr[i]   = res['fraction_metal']

        rows.append({
            'P_up (Pa)':            P_up,
            'J_total (mol/m²/s)':   r['J_total'],
            'J_intact':             J_intact_arr[i],
            'J_pinhole':            J_pinhole_arr[i],
            'J_crack':              J_crack_arr[i],
            'J_grain_boundary':     J_gb_arr[i],
            'frac_intact (%)':      frac_intact_arr[i]  * 100,
            'frac_pinhole (%)':     frac_pinhole_arr[i] * 100,
            'frac_crack (%)':       frac_crack_arr[i]   * 100,
            'frac_GB (%)':          frac_gb_arr[i]      * 100,
            'enhancement_factor':   r['enhancement_vs_intact'],
            'D_eff_avg (m²/s)':     r['D_eff_avg'],
            'D_eff_avg/D_m':        r['D_eff_avg'] / D_m,
            'dominant_path':        dom,
            'fraction_surface (%)': frac_surface_arr[i] * 100
                                    if not np.isnan(frac_surface_arr[i]) else np.nan,
            'fraction_oxide (%)':   frac_oxide_arr[i]   * 100
                                    if not np.isnan(frac_oxide_arr[i])   else np.nan,
            'fraction_metal (%)':   frac_metal_arr[i]   * 100
                                    if not np.isnan(frac_metal_arr[i])   else np.nan,
            'system_rate_limiting': r['system_rate_limiting'].upper(),
            'mode':                 mode,
        })

    except Exception as e:
        dominant_path_arr[i] = 'error'
        rate_lim_arr[i]      = 'error'
        rows.append({
            'P_up (Pa)':           P_up,
            'system_rate_limiting': 'ERROR',
            'mode':                 mode,
            'Error':                str(e),
        })

# ── Derived arrays ────────────────────────────────────────────────────────────
valid = ~np.isnan(J_arr)

# Only classify where flux_weighted_resistances was not None
res_valid = valid & ~np.isnan(frac_surface_arr)

rl_arr = np.where(
    res_valid & (frac_surface_arr > 0.5), 'surface',
    np.where(
        res_valid & (frac_oxide_arr > 0.5), 'oxide',
        np.where(
            res_valid & (frac_metal_arr > 0.5), 'metal', 'mixed'
        )
    )
)

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & res_valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & res_valid)
n_metal   = np.sum((frac_metal_arr   > 0.5) & res_valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) &
    (frac_metal_arr   <= 0.5) & res_valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & res_valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & res_valid
metal_dom_mask = (frac_metal_arr   > 0.90) & res_valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]
P_metal_dom    = P_RANGE_ARR[metal_dom_mask]

P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Crossover pressures ───────────────────────────────────────────────────────
def find_crossover(frac_a, frac_b, valid_mask):
    vc   = valid_mask & ~np.isnan(frac_a) & ~np.isnan(frac_b)
    if np.sum(vc) < 2:
        return None
    diff = frac_a[vc] - frac_b[vc]
    Pv   = P_RANGE_ARR[vc]
    sc   = np.where(np.diff(np.sign(diff)))[0]
    if len(sc) == 0:
        return None
    idx = sc[0]
    return Pv[idx] if diff[idx] > 0 else None

P_cross_surf_oxide  = find_crossover(frac_surface_arr, frac_oxide_arr,  res_valid)
P_cross_surf_metal  = find_crossover(frac_surface_arr, frac_metal_arr,  res_valid)
P_cross_oxide_metal = find_crossover(frac_oxide_arr,   frac_metal_arr,  res_valid)

# ── Dominant path summary ─────────────────────────────────────────────────────
dom_counts = {}
for p in ['intact', 'pinhole', 'crack', 'grain_boundary', 'mixed', 'error']:
    dom_counts[p] = np.sum(dominant_path_arr == p)

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L3+L4+L6 REGIME SUMMARY")
print(f"T = {T_op-273.15:.3f}°C  |  mode = {mode}  |  "
      f"{np.sum(valid)} valid points")
print("=" * 60)

print(f"\n  Resistance regime (dominant path, fraction > 50%)")
print(f"    Surface-limited: {n_surface:3d}  points")
print(f"    Oxide-limited:   {n_oxide:3d}  points")
print(f"    Metal-limited:   {n_metal:3d}  points")
print(f"    Mixed (no res):  {n_mixed:3d}  points")

print(f"\n  Dominant oxide path")
for path, count in dom_counts.items():
    if count > 0:
        print(f"    {path:<18}  {count:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
for label, dom, P_ea in [
    ('Surface', P_surf_dom,  P_for_surface_Ea),
    ('Oxide',   P_oxide_dom, P_for_oxide_Ea),
    ('Metal',   P_metal_dom, P_for_metal_Ea),
]:
    if len(dom) > 3:
        print(f"    {label}-dominant: {len(dom):3d} pts  "
              f"P = {dom.min():.1e} – {dom.max():.1e} Pa  "
              f"→ {P_ea:.2e} Pa")
    else:
        print(f"    {label}-dominant: {len(dom):3d} pts  "
              f"(insufficient — {label.lower()} Ea not extractable)")

print(f"\n  Crossover pressures (diagnostic only)")
for desc, P_co in [
    ('Surface → Oxide', P_cross_surf_oxide),
    ('Surface → Metal', P_cross_surf_metal),
    ('Oxide   → Metal', P_cross_oxide_metal),
]:
    val = f'{P_co:.2e} Pa' if P_co is not None else 'not found'
    print(f"    {desc}:  {val}")

enh_valid = enhancement_arr[valid & ~np.isnan(enhancement_arr)]
if len(enh_valid) > 0:
    print(f"\n  Enhancement (J_total / J_intact):")
    print(f"    Min = {enh_valid.min():.3f}  Max = {enh_valid.max():.3f}")

if np.sum(valid) > 0:
    print(f"\n  J range: {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
else:
    print(f"\n  ⚠  No valid points — check solver signature and config.")
print("=" * 60)

L3+L4+L6 REGIME SUMMARY
T = 599.850°C  |  mode = both  |  40 valid points

  Resistance regime (dominant path, fraction > 50%)
    Surface-limited:  20  points
    Oxide-limited:    18  points
    Metal-limited:     0  points
    Mixed (no res):    2  points

  Dominant oxide path
    intact               27 points
    mixed                13 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  15 pts  P = 1.0e-07 – 1.1e-02 Pa  → 3.26e-05 Pa
    Oxide-dominant:   0 pts  (insufficient — oxide Ea not extractable)
    Metal-dominant:   0 pts  (insufficient — metal Ea not extractable)

  Crossover pressures (diagnostic only)
    Surface → Oxide:  1.51e+00 Pa
    Surface → Metal:  1.51e+00 Pa
    Oxide   → Metal:  not found

  Enhancement (J_total / J_intact):
    Min = 1.032  Max = 2.935

  J range: 6.96e-16 → 8.06e-06 mol/m²/s


In [ ]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation, one figure per panel
#   flux        flux against pressure through the hierarchy L2+L6, L3+L6, L2+L4+L6, L3+L4+L6
#   enhancement each defective level's flux over L2+L6, and whether they combine multiplicatively
#   resistance  flux-weighted resistance fractions of the full model against pressure
#   limit       two limit checks: f→0 recovers L2+L4+L6; mode='none' recovers L3+L6
# The figures are drawn by calculations/surface_figures.py in the house style;
# this cell computes what they show.
# =============================================================================

# ── Flux against pressure: full hierarchy sweep ──────────────────────────────
P_A = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
# Zero-defect config for the f=0 cases
dc_zero = {
    path: {**cfg, 'area_fraction': f_limit_check}
    for path, cfg in defect_config.items()
}
J_curves = {}
fractions_full = None
for solver_key in ('L2L6', 'L3L6', 'L2L4L6', 'L3L4L6'):
    J_h  = np.full(len(P_A), np.nan)
    fs_h = np.full(len(P_A), np.nan)
    fo_h = np.full(len(P_A), np.nan)
    fm_h = np.full(len(P_A), np.nan)
    for i, P_up in enumerate(P_A):
        try:
            if solver_key == 'L2L6':
                r = solve_steady_state_flux_direct(
                    P_up, P_down, L_m,
                    k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m
                )
                J_h[i] = r['J_ss']
            elif solver_key == 'L3L6':
                r = calculate_mixed_defect_flux_L6(
                    P_up, P_down, L_m,
                    k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
                    defect_config,
                    k_diss_metal=k_diss_metal_arg,
                    K_eq_metal=K_eq_metal_arg,
                )
                J_h[i] = r['J_total']
            elif solver_key == 'L2L4L6':
                r = calculate_defective_metal_flux_L6(
                    P_up, P_down, L_m, T_op,
                    k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
                    microstructure_params=microstructure,
                    mode=mode,
                )
                J_h[i] = r['flux']
            elif solver_key == 'L3L4L6':
                r = calculate_full_model_flux_L346_v2(
                    P_up, P_down, L_m, T_op,
                    k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
                    microstructure, defect_config,
                    k_diss_metal=k_diss_metal_arg,
                    K_eq_metal=K_eq_metal_arg,
                    mode=mode,
                )
                J_h[i] = r['J_total']
                res = r.get('flux_weighted_resistances')
                if res is not None:
                    fs_h[i] = res['fraction_surface']
                    fo_h[i] = res['fraction_oxide']
                    fm_h[i] = res['fraction_metal']
        except Exception:
            pass
    J_curves[solver_key] = J_h.copy()
    if solver_key == 'L3L4L6':
        fractions_full = (fs_h, fo_h, fm_h)

# Net slope of the full model
v_net = ~np.isnan(J_arr) & (J_arr > 0)
slope_net = SF.net_slope(P_RANGE_ARR[v_net], J_arr[v_net]) if np.sum(v_net) > 2 else np.nan

# Points whose dominant path is 'mixed' have no flux-weighted resistances
frac_missing = int(np.sum(valid) - np.sum(res_valid))

# ── Limit check: two limits ──────────────────────────────────────────────────
P_D = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
# Limit 1: f→0 should recover L2+L4+L6
J_f0        = np.full(len(P_D), np.nan)   # L3+L4+L6 with f→0
J_L2L4L6_D  = np.full(len(P_D), np.nan)   # L2+L4+L6 reference
# Limit 2: mode='none' should recover L3+L6
J_mnone     = np.full(len(P_D), np.nan)   # L3+L4+L6 with mode=none
J_L3L6_D    = np.full(len(P_D), np.nan)   # L3+L6 reference
for i, P_up in enumerate(P_D):
    try:
        r = calculate_full_model_flux_L346_v2(
            P_up, P_down, L_m, T_op,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            microstructure, dc_zero,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
            mode=mode,
        )
        J_f0[i] = r['J_total']
    except Exception:
        pass
    try:
        r = calculate_defective_metal_flux_L6(
            P_up, P_down, L_m, T_op,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            microstructure_params=microstructure,
            mode=mode,
        )
        J_L2L4L6_D[i] = r['flux']
    except Exception:
        pass
    try:
        r = calculate_full_model_flux_L346_v2(
            P_up, P_down, L_m, T_op,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            microstructure, defect_config,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
            mode='none',
        )
        J_mnone[i] = r['J_total']
    except Exception:
        pass
    try:
        r = calculate_mixed_defect_flux_L6(
            P_up, P_down, L_m,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            defect_config,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
        )
        J_L3L6_D[i] = r['J_total']
    except Exception:
        pass

vp1 = (~np.isnan(J_f0) & ~np.isnan(J_L2L4L6_D) & (J_f0 > 0) & (J_L2L4L6_D > 0))
vp2 = (~np.isnan(J_mnone) & ~np.isnan(J_L3L6_D) & (J_mnone > 0) & (J_L3L6_D > 0))
max_err1 = (np.abs(J_f0[vp1] - J_L2L4L6_D[vp1]) / J_L2L4L6_D[vp1] * 100).max() if np.any(vp1) else np.nan
max_err2 = (np.abs(J_mnone[vp2] - J_L3L6_D[vp2]) / J_L3L6_D[vp2] * 100).max() if np.any(vp2) else np.nan
ok1 = not np.isnan(max_err1) and max_err1 < 1.0
ok2 = not np.isnan(max_err2) and max_err2 < 1.0

# ── Figures ──────────────────────────────────────────────────────────────────
panels = SF.l3l4l6_panels(
    P_A, J_curves, fractions_full, slope_net,
    np.where(res_valid, frac_surface_arr, np.nan), np.where(res_valid, frac_oxide_arr, np.nan),
    np.where(res_valid, frac_metal_arr, np.nan),
    [('S→O', P_cross_surf_oxide), ('O→M', P_cross_oxide_metal)], frac_missing,
    J_f0, J_L2L4L6_D, J_mnone, J_L3L6_D, mode=mode, T_C=T_op - 273.15,
    total_fraction=sum(v['area_fraction'] for v in defect_config.values()),
)
save_panels(panels, "L3L4L6_core_validation", FIGS, module=SF.MODULE)

print("\n" + "=" * 60)
print("L3+L4+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
if np.sum(v_net) > 2:
    print(f"  Net log-log slope (L3+L4+L6): {slope_net:.4f}")
print(f"  D_eff_avg/D_m at T_op:  "
      f"{np.nanmean(D_eff_avg_arr[valid]):.4f}")
print(f"  Enhancement range:  "
      f"{np.nanmin(enhancement_arr[valid]):.3f} – "
      f"{np.nanmax(enhancement_arr[valid]):.3f}")
print(f"\n  Limit check 1 (f→0 → L2+L4+L6): max err = {max_err1:.2e}%  "
      f"{'✓' if ok1 else '⚠'}")
print(f"  Limit check 2 (mode=none → L3+L6): max err = {max_err2:.2e}%  "
      f"{'✓' if ok2 else '⚠'}")

In [78]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('system_rate_limiting', 'dominant_path', 'mode', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower() and 'frac_' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'D_eff' in col and '/D_m' in col:
            df_display[col] = df_display[col].round(6)
        elif 'enhancement' in col.lower():
            df_display[col] = df_display[col].round(4)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['system_rate_limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['system_rate_limiting'] == 'OXIDE').sum()
n_metal_df   = (df['system_rate_limiting'] == 'METAL').sum()
n_mixed_df   = (df['system_rate_limiting'] == 'MIXED').sum()
n_error_df   = (df['system_rate_limiting'] == 'ERROR').sum()

n_intact_df  = (df['dominant_path'] == 'intact').sum()
n_pin_df     = (df['dominant_path'] == 'pinhole').sum()
n_crack_df   = (df['dominant_path'] == 'crack').sum()
n_gb_df      = (df['dominant_path'] == 'grain_boundary').sum()
n_mixdom_df  = (df['dominant_path'] == 'mixed').sum()

print("=" * 60)
print("L3+L4+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:             {OXIDE_KEY}")
print(f"  Metal:             {METAL_KEY}")
print(f"  T:                 {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:              {L_ox*1e6:.3f} μm")
print(f"  L_m:               {L_m*1e3:.3f} mm")
print(f"  mode:              {mode}")
print(f"  grain_size:        {grain_size*1e6:.3f} μm")
print(f"  k_diss:            {k_diss:.3e}  mol/m²/s/Pa")
print(f"  α_intact:          {alpha_intact:.3e}  mol/m²/s/Pa^0.5")
print(f"  β_eff:             {beta_eff:.3e}  mol/m²/s/Pa^0.5  (mode={mode})")
print(f"  D_eff/D_m:         {D_eff/D_m:.6f}")

print(f"\n  Defect configuration")
for path, cfg in defect_config.items():
    print(f"    {path:<16} f = {cfg['area_fraction']*100:.3f}%")

print(f"\n  Total rows:          {len(df)}")
print(f"\n  Resistance regime (dominant path):")
print(f"    Surface-limited:   {n_surface_df}")
print(f"    Oxide-limited:     {n_oxide_df}")
print(f"    Metal-limited:     {n_metal_df}")
print(f"    Mixed (res=None):  {n_mixed_df}")
if n_error_df > 0:
    print(f"    Errors:            {n_error_df}  ⚠")

print(f"\n  Dominant oxide path:")
print(f"    Intact:            {n_intact_df}")
print(f"    Pinhole:           {n_pin_df}")
print(f"    Crack:             {n_crack_df}")
print(f"    Grain boundary:    {n_gb_df}")
print(f"    Mixed (>1 path):   {n_mixdom_df}")

if 'enhancement_factor' in df.columns:
    enh_vals = df['enhancement_factor'].dropna()
    if len(enh_vals) > 0:
        print(f"\n  Enhancement (J_total / J_intact):")
        print(f"    Min  = {enh_vals.min():.4f}")
        print(f"    Max  = {enh_vals.max():.4f}")
        print(f"    Mean = {enh_vals.mean():.4f}")

if 'D_eff_avg/D_m' in df.columns:
    deff_vals = df['D_eff_avg/D_m'].dropna()
    if len(deff_vals) > 0:
        print(f"\n  D_eff_avg/D_m range:  "
              f"{deff_vals.min():.6f} → {deff_vals.max():.6f}")

print("=" * 60)

display(df_display)

L3+L4+L6 RESULTS DATAFRAME
  Oxide:             Cr2O3_sample4
  Metal:             metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:                 599.850°C  (873 K)
  L_ox:              1.000 μm
  L_m:               1.000 mm
  mode:              both
  grain_size:        100.000 μm
  k_diss:            2.370e-09  mol/m²/s/Pa
  α_intact:          4.002e-09  mol/m²/s/Pa^0.5
  β_eff:             7.372e-09  mol/m²/s/Pa^0.5  (mode=both)
  D_eff/D_m:         0.773222

  Defect configuration
    pinhole          f = 1.000%
    crack            f = 0.500%
    grain_boundary   f = 0.500%

  Total rows:          40

  Resistance regime (dominant path):
    Surface-limited:   7
    Oxide-limited:     18
    Metal-limited:     0
    Mixed (res=None):  15

  Dominant oxide path:
    Intact:            27
    Pinhole:           0
    Crack:             0
    Grain boundary:    0
    Mixed (>1 path):   13

  Enhancement (J_total / J_intact):
    Min  = 1.0319
    Max  = 2.9346
    Mean = 1.5557

,P_up (Pa),J_total (mol/m²/s),J_intact,J_pinhole,J_crack,J_grain_boundary,frac_intact (%),frac_pinhole (%),frac_crack (%),frac_GB (%),enhancement_factor,D_eff_avg (m²/s),D_eff_avg/D_m,dominant_path,fraction_surface (%),fraction_oxide (%),fraction_metal (%),system_rate_limiting,mode
0,1.0000e-07,6.9556e-16,2.3702e-16,4.6091e-14,2.3702e-16,2.3702e-16,3.3395e+01,6.6265e+01,1.7038e-01,1.7038e-01,2.9346,1.9227e-10,0.773222,mixed,100.00,0.00,0.00,MIXED,both
1,2.2855e-07,1.5891e-15,5.4170e-16,1.0529e-13,5.4170e-16,5.4170e-16,3.3406e+01,6.6253e+01,1.7044e-01,1.7044e-01,2.9336,1.9227e-10,0.773222,mixed,100.00,0.00,0.00,MIXED,both
2,5.2233e-07,3.6292e-15,1.2380e-15,2.4035e-13,1.2380e-15,1.2380e-15,3.3431e+01,6.6228e+01,1.7057e-01,1.7057e-01,2.9314,1.9227e-10,0.773222,mixed,100.00,0.00,0.00,MIXED,both
3,1.1938e-06,8.2800e-15,2.8295e-15,5.4789e-13,2.8295e-15,2.8295e-15,3.3489e+01,6.6169e+01,1.7086e-01,1.7086e-01,2.9263,1.9227e-10,0.773222,mixed,100.00,0.00,0.00,MIXED,both
4,2.7283e-06,1.8850e-14,6.4667e-15,1.2448e-12,6.4667e-15,6.4667e-15,3.3619e+01,6.6038e+01,1.7153e-01,1.7153e-01,2.9150,1.9227e-10,0.773222,mixed,99.99,0.00,0.01,MIXED,both
5,6.2355e-06,4.2713e-14,1.4779e-14,2.8081e-12,1.4779e-14,1.4779e-14,3.3909e+01,6.5745e+01,1.7301e-01,1.7301e-01,2.8901,1.9227e-10,0.773222,mixed,99.97,0.00,0.03,MIXED,both
6,1.4251e-05,9.5842e-14,3.3777e-14,6.2403e-12,3.3778e-14,3.3778e-14,3.4538e+01,6.5110e+01,1.7621e-01,1.7621e-01,2.8375,1.9227e-10,0.773222,mixed,99.94,0.00,0.06,MIXED,both
7,3.2570e-05,2.1118e-13,7.7195e-14,1.3476e-11,7.7198e-14,7.7198e-14,3.5823e+01,6.3811e+01,1.8278e-01,1.8278e-01,2.7357,1.9227e-10,0.773222,mixed,99.87,0.00,0.13,MIXED,both
8,7.4438e-05,4.5238e-13,1.7642e-13,2.7773e-11,1.7643e-13,1.7643e-13,3.8218e+01,6.1392e+01,1.9500e-01,1.9500e-01,2.5642,1.9227e-10,0.773222,mixed,99.71,0.00,0.29,MIXED,both
9,1.7013e-04,9.3748e-13,4.0317e-13,5.3833e-11,4.0322e-13,4.0322e-13,4.2146e+01,5.7424e+01,2.1506e-01,2.1506e-01,2.3253,1.9227e-10,0.773222,mixed,99.39,0.00,0.61,MIXED,both
